# Meridian health data cleanning project

In [1]:
#import dependices

import pandas as pd
import numpy as np
import re
!pip install rapidfuzz
import rapidfuzz
from rapidfuzz.fuzz import ratio
import networkx as nx
import matplotlib.pyplot as plt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.3 MB/s eta 0:00:00


In [2]:

#read all data files

categories = pd.read_csv("/content/categories.csv")
customers = pd.read_csv("/content/customers_master.csv")
employees = pd.read_csv ("/content/employees.csv")
prescriptions = pd.read_csv("/content/prescriptions_log.csv")
products = pd.read_csv("/content/products_catalog.csv")
stock = pd.read_csv("/content/stock_condition_log.csv")
suppliers = pd.read_csv("/content/suppliers.csv")
stores = pd.read_csv("/content/stores.csv")
sales_g1 = pd.read_csv("/content/sales_group1_storesA.csv")
sales_g2 = pd.read_csv("/content/sales_group2_storesB.csv")
sales_g3 = pd.read_csv("/content/sales_group3_storesC.csv")


# Customers Table

In [3]:
#describe customers table

description = pd.DataFrame({
    "Column": customers.columns,
    "Data Type": customers.dtypes.astype(str).values,
    "Non-Null": customers.notna().sum().values,
    "Null": customers.isna().sum().values,
    "Null %": (customers.isna().mean() * 100).round(2).values,
    "Unique": customers.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,customer_id,int64,1555,0,0.00,1555
1,full_name,object,1555,0,0.00,602
2,email,object,1555,0,0.00,1555
3,phone,object,1555,0,0.00,1513
4,city,object,1555,0,0.00,45
5,signup_date,object,1555,0,0.00,1132
6,notes,object,696,859,55.24,25


In [4]:
#data types change
customers = customers.astype({
    "customer_id": "int64",
    "full_name": "string",
    "email": "string",
    "phone": "string",
    "city": "string",
    "notes": "string"
})

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce"
)

customers.dtypes

,0
customer_id,int64
full_name,string[python]
email,string[python]
phone,string[python]
city,string[python]
signup_date,datetime64[ns]
notes,string[python]


In [5]:
#lowering and striping  string values

customers[customers.select_dtypes(include="string").columns] = (
    customers.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [6]:
customers["city"].unique()

<StringArray>
[     'montreal',      'edmonton',       'moncton',        'regina',
      'waterloo',      'winnipeg',       'toronto',     'vancouver',
    'regina, sk',       'calgary',  'winnipeg, mb',   'quebec city',
     'kitchener',     'saskatoon',        'london',   'calgary, ab',
        'ottawa', 'vancouver, bc',       'halifax',      'vancouvr',
    'ottawa, on',  'montreal, qc',   'halifax, ns',      'hamilton',
   'toronto, on',      'montréal',  'waterloo, on',        'quebec',
  'edmonton, ab',       'qc city',         'trnto',   'moncton, nb']
Length: 32, dtype: string

In [7]:
#mapping city names for remove duplicat meaning

city_mapping = {
    "toronto": "toronto",
    "toronto, on": "toronto",
    "trnto": "toronto",

    "vancouver": "vancouver",
    "vancouver, bc": "vancouver",
    "vancouvr": "vancouver",

    "montreal": "montreal",
    "montréal": "montreal",
    "montreal, qc": "montreal",

    "calgary": "calgary",
    "calgary, ab": "calgary",

    "winnipeg": "winnipeg",
    "winnipeg, mb": "winnipeg",

    "regina": "regina",
    "regina, sk": "regina",

    "ottawa": "ottawa",
    "ottawa, on": "ottawa",

    "halifax": "halifax",
    "halifax, ns": "halifax",

    "edmonton": "edmonton",
    "edmonton, ab": "edmonton",

    "waterloo": "waterloo",
    "waterloo, on": "waterloo",

    "moncton": "moncton",
    "moncton, nb": "moncton",

    "quebec city": "quebec",
    "qc city": "quebec",
    "quebec": "quebec"
}

customers["city"] = customers["city"].replace(city_mapping)

In [8]:
customers["city"].unique()

<StringArray>
[ 'montreal',  'edmonton',   'moncton',    'regina',  'waterloo',  'winnipeg',
   'toronto', 'vancouver',   'calgary',    'quebec', 'kitchener', 'saskatoon',
    'london',    'ottawa',   'halifax',  'hamilton']
Length: 16, dtype: string

In [9]:
# loyalty extaction

def extract_loyalty(note):
    if pd.isna(note) or note == "":
        return pd.Series({"loyalty_tier": None, "loyalty_since": None})

    m = re.search(r"(silver|gold|platinum) member - loyalty since (\d{4})", note)
    if m:
        return pd.Series({"loyalty_tier": m.group(1), "loyalty_since": int(m.group(2))})
    return pd.Series({"loyalty_tier": None, "loyalty_since": None})

loyalty = customers["notes"].apply(extract_loyalty)
loyalty = loyalty.astype({
    "loyalty_tier": "string",
    "loyalty_since": "Int64"
})

customers = pd.concat([customers, loyalty], axis=1)

In [10]:
#phone cleaning
customers["phone"] = customers["phone"].astype(str).str.replace(r"\D", "", regex=True)

In [ ]:
df= customers.copy()
blocks = df.groupby("city").groups

G = nx.Graph()
G.add_nodes_from(df.index)

threshold = 80

for city, indices in blocks.items():
    indices = list(indices)

    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):

            idx_a = indices[i]
            idx_b = indices[j]

            a = df.loc[idx_a]
            b = df.loc[idx_b]

            name_score = ratio(a["full_name"], b["full_name"])
            email_score = ratio(a["email"], b["email"])

            phone_score = (
                100
                if a["phone"] and b["phone"] and a["phone"] == b["phone"]
                else 0
            )

            score = (
                name_score * 0.40 +
                email_score * 0.40 +
                phone_score * 0.20
            )

            if score >= threshold:
                G.add_edge(
                    idx_a,
                    idx_b,
                    score=round(score, 2)
                )


groups = [
    list(group)
    for group in nx.connected_components(G)
    if len(group) > 1
]


results = []

for group_id, group in enumerate(groups, start=1):

    group_scores = []

    for i in range(len(group)):
        for j in range(i + 1, len(group)):

            if G.has_edge(group[i], group[j]):
                score = G[group[i]][group[j]]["score"]
                group_scores.append(score)

    group_score = round(
        sum(group_scores) / len(group_scores),
        2
    )

    for idx in group:
        results.append({
            "group_id": group_id,
            "customer_id": df.loc[idx, "customer_id"],
            "group_size": len(group),
            "group_score": group_score
        })


groups_df = pd.DataFrame(results)




In [ ]:
groups_df

In [ ]:
matched = (
    groups_df
    .merge(
        df[["customer_id", "full_name", "email", "phone", "city"]],
        on="customer_id",
        how="left"
    )
)

result = matched.merge(
    matched,
    on="group_id",
    suffixes=("_1", "_2")
)

result = result[
    result["customer_id_1"] < result["customer_id_2"]
]

result = result[
    [
        "group_id",
        "customer_id_1",
        "customer_id_2",
        "full_name_1",
        "full_name_2",
        "email_1",
        "email_2",
        "phone_1",
        "phone_2",
        "city_1",
        "city_2",
        "group_score_1"
    ]
].rename(
    columns={"group_score_1": "group_score"}
)

result

# Sales Table

In [ ]:

#standeraized the table columns names

sales_g1 = sales_g1.rename(columns={
    "quantity": "quantity",
    "unit_price": "unit_price"
})

sales_g2 = sales_g2.rename(columns={
    "qty": "quantity",
    "amount": "unit_price"
})

sales_g3 = sales_g3.rename(columns={
    "quantity": "quantity",
    "unit_price": "unit_price"
})

# add columns notes into g1 & g2

sales_g1["notes"] = pd.NA
sales_g2["notes"] = pd.NA



In [ ]:
# order columns for each group

columns = [
    "sale_id",
    "store_id",
    "employee_id",
    "product_id",
    "customer_id",
    "quantity",
    "sale_date",
    "unit_price",
    "discount_pct",
    "payment_method",
    "notes"
]

sales_g1 = sales_g1[columns]
sales_g2 = sales_g2[columns]
sales_g3 = sales_g3[columns]

In [ ]:
#combining 3 tables into 1 sales table to start clean

sales = pd.concat(
    [sales_g1, sales_g2, sales_g3],
    ignore_index=True
)

In [ ]:
# validate combination
source_total = len(sales_g1) + len(sales_g2) + len(sales_g3)
combined_total = len(sales)

print("Source Total:", source_total)
print("Combined Total:", combined_total)
print("Match:", source_total == combined_total)

In [ ]:
# describe sales table

description = pd.DataFrame({
    "Column": sales.columns,
    "Data Type": sales.dtypes.astype(str).values,
    "Non-Null": sales.notna().sum().values,
    "Null": sales.isna().sum().values,
    "Null %": (sales.isna().mean() * 100).round(2).values,
    "Unique": sales.nunique().values
})

description

In [ ]:
#data types change

sales["unit_price"] = (
    sales["unit_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

sales["unit_price"] = pd.to_numeric(
    sales["unit_price"],
    errors="coerce"
)

sales = sales.astype({
    "sale_id": "int64",
    "store_id": "int64",
    "employee_id": "int64",
    "product_id": "int64",
    "customer_id": "int64",
    "quantity": "int64",
    "unit_price": "float64",
    "discount_pct": "float64",
    "payment_method": "string",
    "notes": "string"
})

sales["sale_date"] = (
    sales["sale_date"]
    .astype("string")
    .str.strip()
)

sales["sale_date"] = pd.to_datetime(
    sales["sale_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
# describe sales table

description = pd.DataFrame({
    "Column": sales.columns,
    "Data Type": sales.dtypes.astype(str).values,
    "Non-Null": sales.notna().sum().values,
    "Null": sales.isna().sum().values,
    "Null %": (sales.isna().mean() * 100).round(2).values,
    "Unique": sales.nunique().values
})

description

In [ ]:
#lowering and striping  string values

sales[sales.select_dtypes(include="string").columns] = (
    sales.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
def extract_promo(row):
    note = row["notes"]
    discount = row["discount_pct"]

    promo_code = None

    if pd.notna(note) and note.strip() != "":
        m = re.search(r"promo code (\w+) applied at checkout", note)

        if m:
            promo_code = m.group(1)

    # No discount
    if pd.isna(discount) or discount == 0:
        discount_reason = "no_discount"

    # Discount exists → check the reason
    elif pd.notna(note) and note.strip() != "":
        if re.search(r"promo code (\w+) applied at checkout", note):
            discount_reason = "promo_code"

        elif "competitor" in note.lower():
            discount_reason = "price_match"

        elif "loyalty" in note.lower():
            discount_reason = "loyalty_tier"

        else:
            discount_reason = "other"

    # Discount exists but no note
    else:
        discount_reason = "other"

    return pd.Series({
        "promo_code": promo_code,
        "discount_reason": discount_reason
    })


extracted = sales.apply(extract_promo, axis=1)

sales = pd.concat([sales, extracted], axis=1)

sales

In [ ]:
dataframes = {
    "categories": categories,
    "customers": customers,
    "employees": employees,
    "prescriptions": prescriptions,
    "products": products,
    "stock": stock,
    "suppliers": suppliers,
    "stores": stores,
    "sales": sales
}

columns_dict = {name: df.columns.tolist() for name, df in dataframes.items()}

display(columns_dict)

# categories table

In [ ]:
#data types change
categories = categories.astype({
    "category_id": "int64",
    "category_name": "string"
})

In [ ]:
#lowering and striping  string values

categories[categories.select_dtypes(include="string").columns] = (
    categories.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
# describe table

description = pd.DataFrame({
    "Column": categories.columns,
    "Data Type": categories.dtypes.astype(str).values,
    "Non-Null": categories.notna().sum().values,
    "Null": categories.isna().sum().values,
    "Null %": (categories.isna().mean() * 100).round(2).values,
    "Unique": categories.nunique().values
})

description

# employees table

In [ ]:
#data types change
employees = employees.astype({
    "employee_id": "int64",
    "store_id": "int64",
    "employee_name": "string",
    "email": "string",
    "position": "string",
    "salary": "float64",
    "is_pharmacist": "boolean"
})

employees["hire_date"] = (
    employees["hire_date"]
    .astype("string")
    .str.strip()
)

employees["hire_date"] = pd.to_datetime(
    employees["hire_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

employees[employees.select_dtypes(include="string").columns] = (
    employees.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
# describe table

description = pd.DataFrame({
    "Column": employees.columns,
    "Data Type": employees.dtypes.astype(str).values,
    "Non-Null": employees.notna().sum().values,
    "Null": employees.isna().sum().values,
    "Null %": (employees.isna().mean() * 100).round(2).values,
    "Unique": employees.nunique().values
})

description

In [ ]:
# pharmacist handiling
employees["is_pharmacist"] = employees["position"].str.lower().apply(
    lambda x: 1 if x == "pharmacist" else 0
)

employees.loc[employees["is_pharmacist"] == 1]

# prescriptions table

In [ ]:
#data types change
prescriptions = prescriptions.astype({
    "prescription_id": "int64",
    "store_id": "int64",
    "product_id": "int64",
    "customer_id": "int64",
    "doctor_name": "string",
    "dosage": "string"
})

prescriptions["issue_date"] = (
    prescriptions["issue_date"]
    .astype("string")
    .str.strip()
)

prescriptions["issue_date"] = pd.to_datetime(
    prescriptions["issue_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

prescriptions[prescriptions.select_dtypes(include="string").columns] = (
    prescriptions.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": prescriptions.columns,
    "Data Type": prescriptions.dtypes.astype(str).values,
    "Non-Null": prescriptions.notna().sum().values,
    "Null": prescriptions.isna().sum().values,
    "Null %": (prescriptions.isna().mean() * 100).round(2).values,
    "Unique": prescriptions.nunique().values
})

description

#products table



In [ ]:
#data types change
products["unit_price"] = (
    products["unit_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

products["cost_price"] = (
    products["cost_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

products["unit_price"] = pd.to_numeric(
    products["unit_price"],
    errors="coerce"
)

products["cost_price"] = pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)

products = products.astype({
    "product_id": "int64",
    "product_name": "string",
    "category_id": "int64",
    "supplier_id": "int64",
    "unit_price": "float64",
    "cost_price": "float64",
    "brand": "string",
    "storage_condition": "string",
    "pack_info": "string"
})

products["expiry_date"] = (
    products["expiry_date"]
    .astype("string")
    .str.strip()
)

products["expiry_date"] = pd.to_datetime(
    products["expiry_date"],
    format="mixed",
    errors="coerce"
)

products["created_at"] = (
    products["created_at"]
    .astype("string")
    .str.strip()
)

products["created_at"] = pd.to_datetime(
    products["created_at"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

products[products.select_dtypes(include="string").columns] = (
    products.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": products.columns,
    "Data Type": products.dtypes.astype(str).values,
    "Non-Null": products.notna().sum().values,
    "Null": products.isna().sum().values,
    "Null %": (products.isna().mean() * 100).round(2).values,
    "Unique": products.nunique().values
})

description

# stock table

In [ ]:
#data types change
stock = stock.astype({
    "stock_id": "int64",
    "store_id": "int64",
    "product_id": "int64",
    "quantity": "int64",
    "reorder_level": "int64",
    "batch_number": "string",
    "condition_notes": "string"
})

stock["last_restocked"] = (
    stock["last_restocked"]
    .astype("string")
    .str.strip()
)

stock["last_restocked"] = pd.to_datetime(
    stock["last_restocked"],
    format="mixed",
    errors="coerce"
)

stock["expiry_date"] = (
    stock["expiry_date"]
    .astype("string")
    .str.strip()
)

stock["expiry_date"] = pd.to_datetime(
    stock["expiry_date"],
    format="mixed",
    errors="coerce"
)

stock["received_date"] = (
    stock["received_date"]
    .astype("string")
    .str.strip()
)

stock["received_date"] = pd.to_datetime(
    stock["received_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

stock[stock.select_dtypes(include="string").columns] = (
    stock.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": stock.columns,
    "Data Type": stock.dtypes.astype(str).values,
    "Non-Null": stock.notna().sum().values,
    "Null": stock.isna().sum().values,
    "Null %": (stock.isna().mean() * 100).round(2).values,
    "Unique": stock.nunique().values
})

description

# suppliers table

In [ ]:
#data types change
suppliers = suppliers.astype({
    "supplier_id": "int64",
    "supplier_name": "string",
    "contact_person": "string",
    "phone_number": "string",
    "city": "string"
})

In [ ]:
#lowering and striping  string values

suppliers[suppliers.select_dtypes(include="string").columns] = (
    suppliers.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": suppliers.columns,
    "Data Type": suppliers.dtypes.astype(str).values,
    "Non-Null": suppliers.notna().sum().values,
    "Null": suppliers.isna().sum().values,
    "Null %": (suppliers.isna().mean() * 100).round(2).values,
    "Unique": suppliers.nunique().values
})

description

# stores table

In [ ]:
#data types change
stores = stores.astype({
    "store_id": "int64",
    "store_name": "string",
    "region": "string",
    "city": "string",
    "manager_name": "string"
})

In [ ]:
#lowering and striping  string values

stores[stores.select_dtypes(include="string").columns] = (
    stores.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": stores.columns,
    "Data Type": stores.dtypes.astype(str).values,
    "Non-Null": stores.notna().sum().values,
    "Null": stores.isna().sum().values,
    "Null %": (stores.isna().mean() * 100).round(2).values,
    "Unique": stores.nunique().values
})

description

# End of cleaning

# export cleanned data

In [ ]:
# import os
# import zipfile

# tables = {
#     "categories": categories,
#     "customers": customers,
#     "employees": employees,
#     "prescriptions": prescriptions,
#     "products": products,
#     "stock": stock,
#     "suppliers": suppliers,
#     "stores": stores,
#     "sales": sales
# }

# folder = "Meridian Health Cleaned Data"
# os.makedirs(folder, exist_ok=True)

# for name, df in tables.items():
#     df.to_csv(
#         f"{folder}/{name}.csv",
#         index=False
#     )

# with zipfile.ZipFile(
#     "Meridian Health Cleaned Data.zip",
#     "w",
#     zipfile.ZIP_DEFLATED
# ) as zip_file:

#     for file in os.listdir(folder):
#         zip_file.write(
#             f"{folder}/{file}",
#             arcname=file
#         )

In [ ]:
sales = sales.merge(
    products[['product_id', 'cost_price']],
    on='product_id',
    how='left'
)

sales['discount_pct'] = sales['discount_pct'].fillna(0)

sales['total_amount'] = (
    sales['quantity'] * sales['unit_price']
    * (1 - sales['discount_pct'] / 100)
).round(2)

sales['total_cost'] = (
    sales['quantity'] * sales['cost_price']
).round(2)

sales['profit'] = (
    sales['total_amount'] - sales['total_cost']
).round(2)

sales['margin'] = (
    sales['profit'] / sales['total_amount'] * 100
).where(
    sales['total_amount'] != 0
).round(2)

In [ ]:
sales

In [ ]:
!pip install -q  pyngrok plotly streamlit

In [ ]:
%%writefile app.py

import streamlit as st
import plotly.express as px
import pandas as pd

st.set_page_config(
    page_title="Miredian Pharmacies",
    page_icon="💊",
    layout="wide"
)

st.title("Miredian Pharmacies")

st.write("Dashboard is running 🚀")

df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
    "Revenue": [12000, 15000, 13500, 18000, 21000]
})

fig = px.line(
    df,
    x="Month",
    y="Revenue",
    markers=True,
    title="Revenue Trend"
)

fig.update_layout(
    template="plotly_dark"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("2vxWmOORNMRZzEPLGZDLOSGk2yf_6mjNXxHT5SDeyBUz6EFtc")

In [ ]:
from pyngrok import ngrok
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ]
)

time.sleep(3)

public_url = ngrok.connect(8501)

print("Dashboard URL:")
print(public_url)

In [ ]:
%%writefile app.py

import os
import json
import time
import html
import math
import io

import pandas as pd
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px

from geopy.geocoders import Nominatim


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Miredian Pharmacies",
    page_icon="💊",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =========================================================
# GLOBAL CSS
# =========================================================

st.markdown("""
<style>

.stApp {
    background-color: #0b0f19;
}

section[data-testid="stSidebar"] {
    background-color: #111827;
    border-right: 1px solid #1f2937;
}

section[data-testid="stSidebar"] > div {
    padding: 0.8rem;
}

section[data-testid="stSidebar"] [data-testid="column"] {
    padding: 0 !important;
}


/* ================= SIDEBAR ================= */

div.stButton {
    margin: 0;
    padding: 0;
}

div.stButton > button {
    width: 100%;
    min-height: 42px;
    border: none;
    border-radius: 9px;
    background: transparent;
    color: #9ca3af;
    text-align: left;
    padding: 10px 12px;
    margin: 3px 0;
    font-size: 13px;
    font-weight: 500;
    transition: background .18s ease, color .18s ease, transform .18s ease;
}

div.stButton > button:hover {
    background-color: #1f2937;
    color: #f9fafb;
    transform: translateX(2px);
}

div.stButton > button:focus {
    box-shadow: none;
    outline: none;
}

.brand-logo {
    width: 42px;
    height: 42px;
    border-radius: 12px;
    background: linear-gradient(135deg, #0ea5e9, #6366f1);
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-size: 15px;
    font-weight: 800;
    margin-top: 2px;
}

.brand-name {
    color: #f9fafb;
    font-size: 15px;
    font-weight: 700;
    white-space: nowrap;
    line-height: 1.2;
    margin-top: 2px;
}

.brand-subtitle {
    color: #ffffff;
    font-size: 10px;
    white-space: nowrap;
    line-height: 1.2;
    margin-top: 4px;
}

.user-divider {
    height: 1px;
    background-color: #1f2937;
    margin-top: 28px;
    margin-bottom: 15px;
}

.user-avatar {
    width: 38px;
    height: 38px;
    border-radius: 50%;
    background: linear-gradient(135deg, #0ea5e9, #6366f1);
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-size: 12px;
    font-weight: 700;
}

.user-name {
    color: #f9fafb;
    font-size: 12px;
    font-weight: 600;
    white-space: nowrap;
    line-height: 1.3;
    margin-top: 3px;
}

.user-role {
    color: #ffffff;
    font-size: 10px;
    white-space: nowrap;
    line-height: 1.3;
    margin-top: 3px;
}


/* ================= MAIN ================= */

.main-title {
    color: #f9fafb;
    font-size: 29px;
    font-weight: 750;
    margin-bottom: 2px;
}

.main-subtitle {
    color: #ffffff;
    font-size: 13px;
    margin-bottom: 20px;
}


/* ================= SECTION HEADERS ================= */

.section-header {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
    margin-top: 30px;
    margin-bottom: 19px;
}

.section-header::before,
.section-header::after {
    content: "";
    height: 1px;
    flex: 1;
    max-width: 190px;
    background: linear-gradient(
        90deg,
        transparent,
        #22d3ee,
        transparent
    );
    box-shadow: 0 0 8px rgba(34, 211, 238, 0.45);
}

.section-title {
    color: #67e8f9;
    font-size: 18px;
    font-weight: 750;
    letter-spacing: 0.35px;
    text-align: center;
    text-shadow:
        0 0 7px rgba(34, 211, 238, 0.65),
        0 0 18px rgba(34, 211, 238, 0.25);
    white-space: nowrap;
}


/* ================= KPI CARDS ================= */

.kpi-card {
    background: #111827;
    border: 1px solid #1f2937;
    border-radius: 14px;
    padding: 18px;
    min-height: 112px;

    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;

    text-align: center;

    transition:
        border-color .2s ease,
        transform .2s ease;
}

.kpi-card:hover {
    border-color: #334155;
    transform: translateY(-1px);
}

.kpi-label {
    color: #ffffff;
    font-size: 12px;
    font-weight: 600;
    margin-bottom: 7px;
}

.kpi-value {
    color: #f8fafc;
    font-size: 23px;
    font-weight: 750;
    letter-spacing: 0.2px;
    line-height: 1.2;
}

.kpi-sub {
    color: #ffffff;
    font-size: 11px;
    margin-top: 6px;
}


/* ================= CHART TITLES ================= */

.chart-title {
    color: #a5f3fc;
    font-size: 18px;
    font-weight: 750;
    margin-bottom: 5px;
    letter-spacing: 0.15px;
    text-shadow:
        0 0 7px rgba(34, 211, 238, 0.22);
}

.chart-subtitle {
    color: #ffffff;
    font-size: 12px;
    margin-bottom: 8px;
}


/* ================= SELECT DESCRIPTION ================= */

div[data-testid="stSelectbox"] label {
    color: #ffffff !important;
    font-size: 12px !important;
    font-weight: 500 !important;
}


/* ================= FILTERS ================= */

div[data-baseweb="select"] > div {
    background-color: #111827;
    border-color: #273449;
}


/* ================= DIVIDER ================= */

.dashboard-divider {
    height: 1px;
    background: linear-gradient(
        90deg,
        transparent,
        #1f2937,
        transparent
    );
    margin: 28px 0;
}


/* ================= CUSTOMER TOGGLE ================= */

div[data-testid="stToggle"] [role="switch"][aria-checked="true"] {
    background-color: #38BDF8 !important;
    border-color: #38BDF8 !important;
}

div[data-testid="stToggle"] [role="switch"][aria-checked="true"]::after {
    background-color: #ffffff !important;
}

</style>
""", unsafe_allow_html=True)


# =========================================================
# HELPERS
# =========================================================

def section_header(title):

    safe_title = html.escape(str(title))

    st.html(
        f"""
        <div class="section-header">
            <div class="section-title">
                {safe_title}
            </div>
        </div>
        """
    )


def chart_title(title, subtitle=None):

    safe_title = html.escape(str(title))

    st.html(
        f"""
        <div class="chart-title">
            {safe_title}
        </div>
        """
    )

    if subtitle:

        safe_subtitle = html.escape(str(subtitle))

        st.html(
            f"""
            <div class="chart-subtitle">
                {safe_subtitle}
            </div>
            """
        )


def money(value):

    if pd.isna(value):
        value = 0

    return f"${value:,.2f}"


def number(value):

    if pd.isna(value):
        value = 0

    return f"{value:,.2f}"


def percent(value):

    if pd.isna(value):
        value = 0

    return f"{value:,.2f}%"


def dynamic_bar_positions(values, threshold=0.35):

    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).fillna(0)

    if len(values) == 0:
        return []

    max_value = values.max()

    if max_value <= 0:
        return ["outside"] * len(values)

    return [
        "inside"
        if value >= max_value * threshold
        else "outside"
        for value in values
    ]


def bar_text(values):

    return [
        f"{value:,.2f}"
        for value in pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        ).fillna(0)
    ]


def standard_bar_layout(height=330):

    return dict(
        height=height,
        margin=dict(
            l=10,
            r=80,
            t=15,
            b=15
        ),
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#e2e8f0",
            size=12
        ),
        xaxis=dict(
            showgrid=True,
            gridcolor="#243044",
            zeroline=False,
            tickfont=dict(
                color="#dbe4ee",
                size=11
            )
        ),
        yaxis=dict(
            showgrid=False,
            tickfont=dict(
                color="#dbe4ee",
                size=12
            )
        )
    )


# =========================================================
# GEOGRAPHIC HELPERS
# =========================================================

def haversine_distance(lat1, lon1, lat2, lon2):

    """
    Calculate distance between two geographic points in kilometers.
    """

    if any(
        pd.isna(value)
        for value in [
            lat1,
            lon1,
            lat2,
            lon2
        ]
    ):
        return float("nan")

    radius = 6371.0

    lat1 = math.radians(float(lat1))
    lon1 = math.radians(float(lon1))
    lat2 = math.radians(float(lat2))
    lon2 = math.radians(float(lon2))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        +
        math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return radius * c


def nearest_store_analysis(
    customer_geo,
    store_geo
):

    """
    For every customer city, find the geographically nearest
    store city using city-level coordinates.
    """

    if (
        customer_geo.empty
        or store_geo.empty
    ):
        return pd.DataFrame()

    store_points = store_geo[
        [
            "city",
            "lat",
            "lon",
            "Transactions",
            "Revenue",
            "Profit"
        ]
    ].dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()

    customer_points = customer_geo[
        [
            "city",
            "lat",
            "lon",
            "Transactions",
            "Revenue",
            "Profit"
        ]
    ].dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()

    results = []

    for _, customer in customer_points.iterrows():

        best_store = None
        best_distance = float("inf")

        for _, store in store_points.iterrows():

            if (
                customer["lat"] == store["lat"]
                and
                customer["lon"] == store["lon"]
            ):
                distance = 0.0

            else:

                distance = haversine_distance(
                    customer["lat"],
                    customer["lon"],
                    store["lat"],
                    store["lon"]
                )

            if distance < best_distance:

                best_distance = distance
                best_store = store

        if best_store is not None:

            results.append(
                {
                    "Customer City":
                        customer["city"],

                    "Customer Transactions":
                        customer["Transactions"],

                    "Customer Revenue":
                        customer["Revenue"],

                    "Customer Profit":
                        customer["Profit"],

                    "Nearest Store":
                        best_store["city"],

                    "Nearest Store Transactions":
                        best_store["Transactions"],

                    "Nearest Store Revenue":
                        best_store["Revenue"],

                    "Nearest Store Profit":
                        best_store["Profit"],

                    "Distance (km)":
                        best_distance,

                    "lat":
                        customer["lat"],

                    "lon":
                        customer["lon"]
                }
            )

    return pd.DataFrame(results)


def store_overlap_analysis(store_details):

    """
    Find pairs of store locations that are geographically close.
    This works at city-level when only city coordinates are available.
    """

    if len(store_details) < 2:

        return pd.DataFrame()

    records = []

    stores_list = (
        store_details
        .dropna(
            subset=[
                "lat",
                "lon"
            ]
        )
        .reset_index(drop=True)
    )

    for i in range(len(stores_list)):

        store_a = stores_list.iloc[i]

        for j in range(
            i + 1,
            len(stores_list)
        ):

            store_b = stores_list.iloc[j]

            distance = haversine_distance(
                store_a["lat"],
                store_a["lon"],
                store_b["lat"],
                store_b["lon"]
            )

            records.append(
                {
                    "Store A":
                        store_a["store_name"],

                    "City A":
                        store_a["store_city"],

                    "Transactions A":
                        store_a["Transactions"],

                    "Revenue A":
                        store_a["Revenue"],

                    "Store B":
                        store_b["store_name"],

                    "City B":
                        store_b["store_city"],

                    "Transactions B":
                        store_b["Transactions"],

                    "Revenue B":
                        store_b["Revenue"],

                    "Distance (km)":
                        distance,

                    "Combined Transactions":
                        (
                            store_a["Transactions"]
                            +
                            store_b["Transactions"]
                        )
                }
            )

    return pd.DataFrame(records)


# =========================================================
# LOAD DATA
# =========================================================

DATA_PATH = "dashboard_data"

required_files = [
    "categories",
    "customers",
    "employees",
    "prescriptions",
    "products",
    "stock",
    "suppliers",
    "stores",
    "sales"
]

data = {}
missing_files = []

for name in required_files:

    path = os.path.join(
        DATA_PATH,
        f"{name}.parquet"
    )

    if os.path.exists(path):

        data[name] = pd.read_parquet(path)

    else:

        missing_files.append(name)


if missing_files:

    st.error(
        "Missing dashboard data files: "
        + ", ".join(missing_files)
    )

    st.stop()


categories = data["categories"]
customers = data["customers"]
employees = data["employees"]
prescriptions = data["prescriptions"]
products = data["products"]
stock = data["stock"]
suppliers = data["suppliers"]
stores = data["stores"]
sales = data["sales"].copy()


# =========================================================
# PREPARE SALES
# =========================================================

sales["quantity"] = pd.to_numeric(
    sales["quantity"],
    errors="coerce"
).fillna(0)

sales["unit_price"] = pd.to_numeric(
    sales["unit_price"],
    errors="coerce"
).fillna(0)

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
).fillna(0)

sales["cost_price"] = pd.to_numeric(
    sales["cost_price"],
    errors="coerce"
).fillna(0)

sales["total_amount"] = pd.to_numeric(
    sales["total_amount"],
    errors="coerce"
)

sales["total_cost"] = pd.to_numeric(
    sales["total_cost"],
    errors="coerce"
)

sales["profit"] = pd.to_numeric(
    sales["profit"],
    errors="coerce"
)

sales["margin"] = pd.to_numeric(
    sales["margin"],
    errors="coerce"
)

sales["sale_date"] = pd.to_datetime(
    sales["sale_date"],
    errors="coerce"
)


# =========================================================
# FINANCIAL FALLBACKS
# =========================================================

calculated_revenue = (
    sales["quantity"]
    * sales["unit_price"]
    * (1 - sales["discount_pct"] / 100)
)

calculated_cost = (
    sales["quantity"]
    * sales["cost_price"]
)


sales["total_amount"] = (
    sales["total_amount"]
    .fillna(calculated_revenue)
    .round(2)
)

sales["total_cost"] = (
    sales["total_cost"]
    .fillna(calculated_cost)
    .round(2)
)

sales["profit"] = (
    sales["profit"]
    .fillna(
        sales["total_amount"]
        - sales["total_cost"]
    )
    .round(2)
)


# =========================================================
# DISCOUNT
# =========================================================

sales["gross_amount"] = (
    sales["quantity"]
    * sales["unit_price"]
).round(2)

sales["discount_amount"] = (
    sales["gross_amount"]
    - sales["total_amount"]
).clip(
    lower=0
).round(2)


# =========================================================
# PRODUCT / CATEGORY
# =========================================================

product_lookup = products[
    [
        "product_id",
        "category_id",
        "product_name"
    ]
].copy()


sales = sales.merge(
    product_lookup,
    on="product_id",
    how="left",
    suffixes=("", "_product")
)


category_lookup = categories[
    [
        "category_id",
        "category_name"
    ]
].copy()


sales = sales.merge(
    category_lookup,
    on="category_id",
    how="left"
)


sales["category_name"] = (
    sales["category_name"]
    .fillna("Unknown")
)


# =========================================================
# STORE
# =========================================================

store_lookup = stores[
    [
        "store_id",
        "store_name",
        "city",
        "region"
    ]
].copy()


store_lookup = store_lookup.rename(
    columns={
        "city": "store_city"
    }
)


sales = sales.merge(
    store_lookup,
    on="store_id",
    how="left"
)


sales["store_name"] = (
    sales["store_name"]
    .fillna("Unknown")
)

sales["store_city"] = (
    sales["store_city"]
    .fillna("Unknown")
)


# =========================================================
# CUSTOMER / LOYALTY
# =========================================================

customer_lookup = customers[
    [
        "customer_id",
        "city",
        "loyalty_tier"
    ]
].copy()


customer_lookup = customer_lookup.rename(
    columns={
        "city": "customer_city"
    }
)


sales = sales.merge(
    customer_lookup,
    on="customer_id",
    how="left"
)


sales["customer_city"] = (
    sales["customer_city"]
    .fillna("Unknown")
)

sales["loyalty_tier"] = (
    sales["loyalty_tier"]
    .fillna("No Loyalty")
)


# =========================================================
# GEOLOCATION
# =========================================================

GEO_CACHE_FILE = os.path.join(
    DATA_PATH,
    "city_coordinates.json"
)


def load_geo_cache():

    if os.path.exists(GEO_CACHE_FILE):

        try:

            with open(
                GEO_CACHE_FILE,
                "r",
                encoding="utf-8"
            ) as f:

                return json.load(f)

        except Exception:

            return {}

    return {}


def save_geo_cache(cache):

    try:

        with open(
            GEO_CACHE_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                cache,
                f,
                ensure_ascii=False,
                indent=2
            )

    except Exception:

        pass


@st.cache_data(ttl=86400)
def geocode_cities(city_list):

    cache = load_geo_cache()

    geolocator = Nominatim(
        user_agent="miredian-pharmacies-dashboard"
    )

    result = {}

    for city in city_list:

        city_key = str(city).strip()

        if not city_key or city_key == "Unknown":
            continue

        if city_key in cache:

            result[city_key] = cache[city_key]
            continue

        try:

            location = geolocator.geocode(
                city_key,
                timeout=10
            )

            if location:

                coords = {
                    "lat": float(location.latitude),
                    "lon": float(location.longitude)
                }

                cache[city_key] = coords
                result[city_key] = coords

                save_geo_cache(cache)

                time.sleep(1)

        except Exception:

            continue

    return result


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    logo_col, brand_col = st.columns(
        [0.22, 0.78],
        gap="small"
    )

    with logo_col:

        st.markdown(
            '<div class="brand-logo">MP</div>',
            unsafe_allow_html=True
        )

    with brand_col:

        st.markdown(
            """
            <div class="brand-name">
                Miredian Pharmacies
            </div>
            <div class="brand-subtitle">
                Management Dashboard
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        '<div class="user-divider"></div>',
        unsafe_allow_html=True
    )


    pages = [
        ("📊", "Overview"),
        ("💰", "Sales"),
        ("👥", "Customer Loyalty"),
        ("📦", "Inventory"),
        ("👨‍💼", "Employees"),
        ("🚚", "Suppliers")
    ]


    if "current_page" not in st.session_state:

        st.session_state.current_page = "Overview"


    for icon, page_name in pages:

        if st.button(
            f"{icon}   {page_name}",
            key=f"nav_{page_name}"
        ):

            st.session_state.current_page = page_name
            st.rerun()


    st.markdown(
        '<div class="user-divider"></div>',
        unsafe_allow_html=True
    )


    avatar_col, user_col = st.columns(
        [0.22, 0.78],
        gap="small"
    )


    with avatar_col:

        st.markdown(
            '<div class="user-avatar">OO</div>',
            unsafe_allow_html=True
        )


    with user_col:

        st.markdown(
            """
            <div class="user-name">
                Omar Osama
            </div>
            <div class="user-role">
                System Admin
            </div>
            """,
            unsafe_allow_html=True
        )


page = st.session_state.current_page

# =========================================================
# OVERVIEW
# =========================================================

if page == "Overview":

    st.markdown(
        '<div class="main-title">Overview</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="main-subtitle">'
        'Miredian Pharmacies · Business Performance'
        '</div>',
        unsafe_allow_html=True
    )


    # =====================================================
    # ANALYSIS PERIOD
    # =====================================================

    analysis_start = pd.Timestamp("2024-10-01")
    analysis_end = pd.Timestamp("2025-12-31")


    # =====================================================
    # COPY DATA
    # =====================================================

    overview_sales = sales.copy()
    overview_stock = stock.copy()
    overview_products = products.copy()
    overview_employees = employees.copy()


    # =====================================================
    # DATE CONVERSION
    # =====================================================

    overview_sales["sale_date"] = pd.to_datetime(
        overview_sales["sale_date"],
        errors="coerce"
    )

    overview_stock["received_date"] = pd.to_datetime(
        overview_stock["received_date"],
        errors="coerce"
    )

    overview_stock["expiry_date"] = pd.to_datetime(
        overview_stock["expiry_date"],
        errors="coerce"
    )

    overview_employees["hire_date"] = pd.to_datetime(
        overview_employees["hire_date"],
        errors="coerce"
    )


    # =====================================================
    # SALES - ANALYSIS PERIOD
    # =====================================================

    overview_sales = overview_sales[
        overview_sales["sale_date"].notna()
        & (overview_sales["sale_date"] >= analysis_start)
        & (overview_sales["sale_date"] <= analysis_end)
    ].copy()


    # =====================================================
    # SALES FINANCIAL DATA
    # =====================================================

    # لو sales عنده already calculated financial columns
    # هنستخدمها مباشرة.
    #
    # ولو مش موجودة، نحسبها هنا.

    if "total_amount" not in overview_sales.columns:

        overview_sales["unit_price"] = pd.to_numeric(
            overview_sales["unit_price"],
            errors="coerce"
        ).fillna(0)

        overview_sales["quantity"] = pd.to_numeric(
            overview_sales["quantity"],
            errors="coerce"
        ).fillna(0)

        overview_sales["discount_pct"] = pd.to_numeric(
            overview_sales["discount_pct"],
            errors="coerce"
        ).fillna(0)

        overview_sales["gross_amount"] = (
            overview_sales["quantity"]
            * overview_sales["unit_price"]
        )

        overview_sales["discount_amount"] = (
            overview_sales["gross_amount"]
            * overview_sales["discount_pct"]
            / 100
        )

        overview_sales["total_amount"] = (
            overview_sales["gross_amount"]
            - overview_sales["discount_amount"]
        )


    # =====================================================
    # COST OF SOLD PRODUCTS
    # =====================================================

    if "total_cost" not in overview_sales.columns:

        # Remove any existing cost_price to avoid
        # cost_price_x / cost_price_y problems.
        if "cost_price" in overview_sales.columns:
            overview_sales = overview_sales.drop(
                columns=["cost_price"]
            )

        product_cost = overview_products[
            ["product_id", "cost_price"]
        ].copy()

        product_cost["cost_price"] = pd.to_numeric(
            product_cost["cost_price"],
            errors="coerce"
        ).fillna(0)

        overview_sales = overview_sales.merge(
            product_cost,
            on="product_id",
            how="left"
        )

        overview_sales["cost_price"] = (
            pd.to_numeric(
                overview_sales["cost_price"],
                errors="coerce"
            )
            .fillna(0)
        )

        overview_sales["total_cost"] = (
            overview_sales["quantity"]
            * overview_sales["cost_price"]
        )


    # =====================================================
    # PROFIT
    # =====================================================

    if "profit" not in overview_sales.columns:

        overview_sales["profit"] = (
            overview_sales["total_amount"]
            - overview_sales["total_cost"]
        )


    # Make sure numerical columns are numeric

    overview_sales["total_amount"] = pd.to_numeric(
        overview_sales["total_amount"],
        errors="coerce"
    ).fillna(0)

    overview_sales["total_cost"] = pd.to_numeric(
        overview_sales["total_cost"],
        errors="coerce"
    ).fillna(0)

    overview_sales["profit"] = pd.to_numeric(
        overview_sales["profit"],
        errors="coerce"
    ).fillna(0)


    # =====================================================
    # FINANCIAL KPIs
    # =====================================================

    total_revenue = overview_sales["total_amount"].sum()

    total_cost = overview_sales["total_cost"].sum()

    sales_profit = overview_sales["profit"].sum()

    sales_margin = (
        sales_profit / total_revenue * 100
        if total_revenue != 0
        else 0
    )


    # =====================================================
    # INVENTORY RECEIVED DURING PERIOD
    # =====================================================

    period_stock = overview_stock[
        overview_stock["received_date"].notna()
        & (overview_stock["received_date"] >= analysis_start)
        & (overview_stock["received_date"] <= analysis_end)
    ].copy()


    # =====================================================
    # INVENTORY COST
    # =====================================================

    # Only merge products if stock doesn't already
    # contain cost_price.

    if "cost_price" not in period_stock.columns:

        product_cost = overview_products[
            ["product_id", "cost_price"]
        ].copy()

        product_cost["cost_price"] = pd.to_numeric(
            product_cost["cost_price"],
            errors="coerce"
        ).fillna(0)

        period_stock = period_stock.merge(
            product_cost,
            on="product_id",
            how="left"
        )

    else:

        period_stock["cost_price"] = pd.to_numeric(
            period_stock["cost_price"],
            errors="coerce"
        ).fillna(0)


    # =====================================================
    # INVENTORY QUANTITY
    # =====================================================

    period_stock["quantity"] = pd.to_numeric(
        period_stock["quantity"],
        errors="coerce"
    ).fillna(0)


    # =====================================================
    # INVENTORY VALUE
    # =====================================================

    period_stock["inventory_cost"] = (
        period_stock["quantity"]
        * period_stock["cost_price"]
    )


    # =====================================================
    # EXPIRED INVENTORY
    # =====================================================

    period_stock["is_expired"] = (
        period_stock["expiry_date"].notna()
        & (
            period_stock["expiry_date"]
            <= analysis_end
        )
    )

    expired_stock = period_stock[
        period_stock["is_expired"]
    ].copy()

    expired_inventory_cost = (
        expired_stock["inventory_cost"].sum()
    )


    # =====================================================
    # EXPIRED ON ARRIVAL
    # =====================================================

    period_stock["is_expired_on_arrival"] = (
        period_stock["received_date"].notna()
        & period_stock["expiry_date"].notna()
        & (
            period_stock["received_date"]
            > period_stock["expiry_date"]
        )
    )

    expired_on_arrival = period_stock[
        period_stock["is_expired_on_arrival"]
    ].copy()

    expired_on_arrival_cost = (
        expired_on_arrival["inventory_cost"].sum()
    )


    # =====================================================
    # AT-RISK INVENTORY
    # =====================================================

    risk_end_date = (
        analysis_end
        + pd.DateOffset(months=3)
    )

    if "condition_notes" in period_stock.columns:

        period_stock["condition_notes"] = (
            period_stock["condition_notes"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        condition_risk = (
            period_stock["condition_notes"]
            .str.contains(
                "risk|at risk|damaged|damage|near expiry",
                regex=True,
                na=False
            )
        )

    else:

        condition_risk = pd.Series(
            False,
            index=period_stock.index
        )


    expiry_risk = (
        period_stock["expiry_date"].notna()
        & (
            period_stock["expiry_date"]
            > analysis_end
        )
        & (
            period_stock["expiry_date"]
            <= risk_end_date
        )
    )

    period_stock["is_at_risk"] = (
        condition_risk
        | expiry_risk
    )


    # Exclude already expired items from risk
    at_risk_stock = period_stock[
        period_stock["is_at_risk"]
        & ~period_stock["is_expired"]
    ].copy()

    at_risk_inventory_cost = (
        at_risk_stock["inventory_cost"].sum()
    )


    # =====================================================
    # TOTAL INVENTORY RECEIVED
    # =====================================================

    total_inventory_received = (
        period_stock["inventory_cost"].sum()
    )


    # =====================================================
    # SALARY COST
    # =====================================================

    # Employees hired on or before analysis end.
    #
    # Employee hired before analysis_start:
    # -> counted for all 15 months.
    #
    # Employee hired during analysis period:
    # -> counted from hire month until analysis_end.

    period_employees = overview_employees[
        overview_employees["hire_date"].notna()
        & (
            overview_employees["hire_date"]
            <= analysis_end
        )
    ].copy()


    period_employees["employment_start"] = (
        period_employees["hire_date"].where(
            period_employees["hire_date"] > analysis_start,
            analysis_start
        )
    )


    period_employees["months_in_period"] = (
        (
            analysis_end.dt.year
            if hasattr(analysis_end, "dt")
            else analysis_end.year
        )
    )


    # Calculate number of months directly

    period_employees["months_in_period"] = (
        (
            analysis_end.year
            - period_employees["employment_start"].dt.year
        )
        * 12
        +
        (
            analysis_end.month
            - period_employees["employment_start"].dt.month
        )
        + 1
    )


    period_employees["salary"] = pd.to_numeric(
        period_employees["salary"],
        errors="coerce"
    ).fillna(0)


    period_employees["salary_cost"] = (
        period_employees["salary"]
        * period_employees["months_in_period"]
    )


    total_salary_cost = (
        period_employees["salary_cost"].sum()
    )


    # =====================================================
    # ADJUSTED PROFIT
    # =====================================================

    adjusted_profit = (
        sales_profit
        - expired_inventory_cost
        - total_salary_cost
    )


    adjusted_margin = (
        adjusted_profit / total_revenue * 100
        if total_revenue != 0
        else 0
    )


    # =====================================================
    # CUSTOMER KPIs
    # =====================================================

    if "customer_id" in overview_sales.columns:

        active_customers = (
            overview_sales["customer_id"]
            .dropna()
            .nunique()
        )

    else:

        active_customers = 0


    # =====================================================
    # SALES VOLUME
    # =====================================================

    total_units = (
        pd.to_numeric(
            overview_sales["quantity"],
            errors="coerce"
        )
        .fillna(0)
        .sum()
    )


    # =====================================================
    # TRANSACTIONS
    # =====================================================

    if "sale_id" in overview_sales.columns:

        total_transactions = (
            overview_sales["sale_id"]
            .nunique()
        )

    else:

        total_transactions = len(
            overview_sales
        )


    # =====================================================
    # AVERAGE ORDER VALUE
    # =====================================================

    average_order_value = (
        total_revenue / total_transactions
        if total_transactions != 0
        else 0
    )


    # =====================================================
    # SECTION HEADER
    # =====================================================

    section_header(
        "Business Performance"
    )


    # =====================================================
    # KPI CARDS
    # =====================================================

    col1, col2, col3, col4 = st.columns(4)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Revenue</div>
                <div class="kpi-value">
                    {money(total_revenue)}
                </div>
                <div class="kpi-subtitle">
                    Oct 2024 – Dec 2025
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Sales Profit</div>
                <div class="kpi-value">
                    {money(sales_profit)}
                </div>
                <div class="kpi-subtitle">
                    {percent(sales_margin)} margin
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Adjusted Profit</div>
                <div class="kpi-value">
                    {money(adjusted_profit)}
                </div>
                <div class="kpi-subtitle">
                    After inventory loss & salaries
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col4:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Adjusted Margin</div>
                <div class="kpi-value">
                    {percent(adjusted_margin)}
                </div>
                <div class="kpi-subtitle">
                    {number(active_customers)} active customers
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # =====================================================
    # MONTHLY PERFORMANCE
    # =====================================================

    monthly = (
        overview_sales
        .groupby(
            overview_sales["sale_date"].dt.to_period("M")
        )
        .agg(
            Revenue=("total_amount", "sum"),
            Profit=("profit", "sum")
        )
        .reset_index()
    )

    monthly["Month"] = (
        monthly["sale_date"]
        .dt
        .to_timestamp()
    )


    # =====================================================
    # CATEGORY PERFORMANCE
    # =====================================================

    if (
        "category_name" not in overview_sales.columns
        and "category_id" in overview_sales.columns
    ):

        category_lookup = overview_products[
            ["product_id", "category_id"]
        ].merge(
            categories[
                ["category_id", "category_name"]
            ],
            on="category_id",
            how="left"
        )

        overview_sales = overview_sales.merge(
            category_lookup[
                ["product_id", "category_name"]
            ],
            on="product_id",
            how="left"
        )


    if "category_name" in overview_sales.columns:

        category_data = (
            overview_sales
            .groupby("category_name")
            .agg(
                Revenue=("total_amount", "sum"),
                Profit=("profit", "sum")
            )
            .reset_index()
            .sort_values(
                "Revenue",
                ascending=False
            )
        )

    else:

        category_data = pd.DataFrame(
            columns=[
                "category_name",
                "Revenue",
                "Profit"
            ]
        )


    # =====================================================
    # CHARTS
    # =====================================================

    chart_col1, chart_col2 = st.columns(
        [1.55, 1]
    )


    # =====================================================
    # MONTHLY REVENUE / PROFIT
    # =====================================================

    with chart_col1:

        chart_title(
            "Revenue & Profit Trend"
        )

        fig_monthly = go.Figure()


        if not monthly.empty:

            fig_monthly.add_trace(
                go.Bar(
                    x=monthly["Month"],
                    y=monthly["Revenue"],
                    name="Revenue",
                    text=[
                        money(x)
                        for x in monthly["Revenue"]
                    ],
                    textposition="outside"
                )
            )


            fig_monthly.add_trace(
                go.Scatter(
                    x=monthly["Month"],
                    y=monthly["Profit"],
                    name="Profit",
                    mode="lines+markers",
                    text=[
                        money(x)
                        for x in monthly["Profit"]
                    ],
                    hovertemplate=(
                        "%{x|%b %Y}"
                        "<br>Profit: %{text}"
                        "<extra></extra>"
                    )
                )
            )


        fig_monthly.update_layout(
            height=360,
            margin=dict(
                l=10,
                r=10,
                t=25,
                b=10
            ),
            xaxis=dict(
                title=None,
                tickformat="%b %Y"
            ),
            yaxis=dict(
                title=None,
                showgrid=True
            ),
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            ),
            hovermode="x unified"
        )


        st.plotly_chart(
            fig_monthly,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # CATEGORY CHART
    # =====================================================

    with chart_col2:

        chart_title(
            "Revenue by Category"
        )

        fig_category = go.Figure()


        if not category_data.empty:

            category_data = category_data.sort_values(
                "Revenue",
                ascending=True
            )


            fig_category.add_trace(
                go.Bar(
                    x=category_data["Revenue"],
                    y=category_data["category_name"],
                    orientation="h",
                    text=[
                        money(x)
                        for x in category_data["Revenue"]
                    ],
                    textposition="outside",
                    hovertemplate=(
                        "%{y}"
                        "<br>Revenue: %{text}"
                        "<extra></extra>"
                    )
                )
            )


        fig_category.update_layout(
            height=360,
            margin=dict(
                l=10,
                r=45,
                t=25,
                b=10
            ),
            xaxis=dict(
                title=None,
                showgrid=True
            ),
            yaxis=dict(
                title=None
            ),
            showlegend=False
        )


        st.plotly_chart(
            fig_category,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # PROFITABILITY / INVENTORY SECTION
    # =====================================================

    section_header(
        "Profitability & Inventory"
    )


    info_col1, info_col2, info_col3, info_col4 = st.columns(4)


    with info_col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Inventory
                </div>
                <div class="kpi-value">
                    {money(expired_inventory_cost)}
                </div>
                <div class="kpi-subtitle">
                    Realized inventory loss
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with info_col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired on Arrival
                </div>
                <div class="kpi-value">
                    {money(expired_on_arrival_cost)}
                </div>
                <div class="kpi-subtitle">
                    Included in expired inventory
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with info_col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Inventory
                </div>
                <div class="kpi-value">
                    {money(at_risk_inventory_cost)}
                </div>
                <div class="kpi-subtitle">
                    Potential future exposure
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with info_col4:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Salary Cost
                </div>
                <div class="kpi-value">
                    {money(total_salary_cost)}
                </div>
                <div class="kpi-subtitle">
                    Across the analysis period
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # =====================================================
    # PERIOD SUMMARY
    # =====================================================

    st.markdown(
        "<br>",
        unsafe_allow_html=True
    )


    summary_col1, summary_col2, summary_col3 = st.columns(3)


    with summary_col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Inventory Received
                </div>
                <div class="kpi-value">
                    {money(total_inventory_received)}
                </div>
                <div class="kpi-subtitle">
                    Stock received during period
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with summary_col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Units Sold
                </div>
                <div class="kpi-value">
                    {number(total_units)}
                </div>
                <div class="kpi-subtitle">
                    Total units sold
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with summary_col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Average Order Value
                </div>
                <div class="kpi-value">
                    {money(average_order_value)}
                </div>
                <div class="kpi-subtitle">
                    Revenue per transaction
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )
# SALES
# =========================================================

elif page == "Sales":

    st.markdown(
        '<div class="main-title">Sales</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="main-subtitle">'
        'Sales Performance · Revenue · Profitability · Discounts'
        '</div>',
        unsafe_allow_html=True
    )


    # =====================================================
    # GENERAL FILTERS
    # =====================================================

    section_header(
        "Filters"
    )

    filter_col1, filter_col2, filter_col3, filter_col4 = st.columns(4)


    with filter_col1:

        sales_store_options = [
            "All Stores"
        ] + sorted(
            sales["store_name"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )

        selected_sales_store = st.selectbox(
            "Store",
            sales_store_options,
            key="sales_filter_store"
        )


    with filter_col2:

        sales_category_options = [
            "All Categories"
        ] + sorted(
            sales["category_name"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )

        selected_sales_category = st.selectbox(
            "Category",
            sales_category_options,
            key="sales_filter_category"
        )


    with filter_col3:

        sales_payment_options = [
            "All Payment Methods"
        ] + sorted(
            sales["payment_method"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )

        selected_sales_payment = st.selectbox(
            "Payment Method",
            sales_payment_options,
            key="sales_filter_payment"
        )


    with filter_col4:

        sales_city_options = [
            "All Cities"
        ] + sorted(
            sales["store_city"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )

        selected_sales_city = st.selectbox(
            "City",
            sales_city_options,
            key="sales_filter_city"
        )


    # =====================================================
    # APPLY GENERAL FILTERS
    # =====================================================

    filtered_sales = sales.copy()


    if selected_sales_store != "All Stores":

        filtered_sales = filtered_sales[
            filtered_sales["store_name"]
            .astype(str)
            .str.strip()
            == selected_sales_store
        ]


    if selected_sales_category != "All Categories":

        filtered_sales = filtered_sales[
            filtered_sales["category_name"]
            .astype(str)
            .str.strip()
            == selected_sales_category
        ]


    if selected_sales_payment != "All Payment Methods":

        filtered_sales = filtered_sales[
            filtered_sales["payment_method"]
            .astype(str)
            .str.strip()
            == selected_sales_payment
        ]


    if selected_sales_city != "All Cities":

        filtered_sales = filtered_sales[
            filtered_sales["store_city"]
            .astype(str)
            .str.strip()
            == selected_sales_city
        ]


    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    # =====================================================
    # FINANCIAL OVERVIEW
    # =====================================================

    section_header(
        "Financial Overview"
    )


    total_revenue = (
        filtered_sales["total_amount"]
        .sum()
    )


    total_profit = (
        filtered_sales["profit"]
        .sum()
    )


    total_cost = (
        filtered_sales["total_cost"]
        .sum()
    )


    profit_margin = (
        total_profit
        / total_revenue
        * 100
        if total_revenue != 0
        else 0
    )


    transaction_count = (
        filtered_sales["sale_id"]
        .nunique()
    )


    c1, c2, c3, c4 = st.columns(4)


    financial_cards = [
        (
            "Total Revenue",
            money(total_revenue),
            f"{transaction_count:,.2f} transactions"
        ),
        (
            "Total Profit",
            money(total_profit),
            "Revenue − Cost"
        ),
        (
            "Profit Margin",
            percent(profit_margin),
            "Weighted margin"
        ),
        (
            "Total Cost",
            money(total_cost),
            "Cost of goods sold"
        )
    ]


    for col, card in zip(
        [c1, c2, c3, c4],
        financial_cards
    ):

        with col:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        {card[0]}
                    </div>
                    <div class="kpi-value">
                        {card[1]}
                    </div>
                    <div class="kpi-sub">
                        {card[2]}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


    # =====================================================
    # ORDERS
    # =====================================================

    section_header(
        "Order Performance"
    )


    aov = (
        total_revenue
        / transaction_count
        if transaction_count != 0
        else 0
    )


    c1, c2 = st.columns(2)


    with c1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Average Order Value
                </div>
                <div class="kpi-value">
                    {money(aov)}
                </div>
                <div class="kpi-sub">
                    Average revenue per order
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with c2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Orders
                </div>
                <div class="kpi-value">
                    {transaction_count:,.2f}
                </div>
                <div class="kpi-sub">
                    Unique sales transactions
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # =====================================================
    # DISCOUNTS
    # =====================================================

    section_header(
        "Discount Performance"
    )


    total_discount = (
        filtered_sales["discount_amount"]
        .sum()
    )


    profit_before_discount = (
        filtered_sales["gross_amount"].sum()
        - filtered_sales["total_cost"].sum()
    )


    discount_pct_of_profit = (
        total_discount
        / profit_before_discount
        * 100
        if profit_before_discount != 0
        else 0
    )


    avg_discount = (
        filtered_sales["discount_pct"].mean()
        if len(filtered_sales) > 0
        else 0
    )


    c1, c2, c3 = st.columns(3)


    discount_cards = [
        (
            "Total Discount Amount",
            money(total_discount),
            "Absolute discount value"
        ),
        (
            "Discount % of Profit",
            percent(discount_pct_of_profit),
            "Discount ÷ profit before discount"
        ),
        (
            "Average Discount",
            percent(avg_discount),
            "Average discount rate"
        )
    ]


    for col, card in zip(
        [c1, c2, c3],
        discount_cards
    ):

        with col:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        {card[0]}
                    </div>
                    <div class="kpi-value">
                        {card[1]}
                    </div>
                    <div class="kpi-sub">
                        {card[2]}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


    # =====================================================
    # VOLUME
    # =====================================================

    section_header(
        "Volume Metrics"
    )


    total_quantity = (
        filtered_sales["quantity"]
        .sum()
    )


    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">
                Total Quantities Sold
            </div>
            <div class="kpi-value">
                {number(total_quantity)}
            </div>
            <div class="kpi-sub">
                Total units sold
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


    # =====================================================
    # CATEGORY / CITY
    # =====================================================

    section_header(
        "Category & City Performance"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Category Performance",
            "Compare categories by the selected metric"
        )


        category_metric = st.selectbox(
            "Select the metric to compare categories",
            [
                "Revenue",
                "Profit",
                "Profit Margin",
                "Total Cost"
            ],
            key="category_metric"
        )


        category_group = (
            filtered_sales
            .groupby(
                "category_name",
                dropna=False
            )
            .agg(
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                ),
                Total_Cost=(
                    "total_cost",
                    "sum"
                )
            )
            .reset_index()
        )


        category_group["Profit Margin"] = (
            category_group["Profit"]
            / category_group["Revenue"]
            * 100
        ).where(
            category_group["Revenue"] != 0
        ).fillna(0)


        metric_column_map = {
            "Revenue": "Revenue",
            "Profit": "Profit",
            "Profit Margin": "Profit Margin",
            "Total Cost": "Total_Cost"
        }


        selected_column = metric_column_map[
            category_metric
        ]


        category_group = (
            category_group
            .sort_values(
                selected_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            category_group[selected_column]
        )


        fig = go.Figure(
            go.Bar(
                x=category_group[selected_column],
                y=category_group["category_name"],
                orientation="h",
                text=bar_text(
                    category_group[selected_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(350)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "City Performance",
            "Compare store cities by the selected metric"
        )


        city_metric = st.selectbox(
            "Select the metric to compare cities",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin",
                "Cost"
            ],
            key="city_metric"
        )


        city_group = (
            filtered_sales
            .groupby(
                "store_city",
                dropna=False
            )
            .agg(
                Transaction_Count=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                ),
                Cost=(
                    "total_cost",
                    "sum"
                )
            )
            .reset_index()
        )


        city_group["Profit Margin"] = (
            city_group["Profit"]
            / city_group["Revenue"]
            * 100
        ).where(
            city_group["Revenue"] != 0
        ).fillna(0)


        city_column_map = {
            "Transaction Count":
                "Transaction_Count",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin",
            "Cost":
                "Cost"
        }


        city_column = city_column_map[
            city_metric
        ]


        city_group = (
            city_group
            .sort_values(
                city_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            city_group[city_column]
        )


        fig = go.Figure(
            go.Bar(
                x=city_group[city_column],
                y=city_group["store_city"],
                orientation="h",
                text=bar_text(
                    city_group[city_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(350)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # PROMO + LOYALTY
    # =====================================================

    section_header(
        "Promo Codes & Customer Loyalty"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Promo Code Analytics",
            "Compare promo code performance by the selected metric"
        )


        promo_metric = st.selectbox(
            "Select the metric to compare promo codes",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="promo_metric"
        )


        promo_sales = filtered_sales.copy()


        promo_sales["promo_display"] = (
            promo_sales["promo_code"]
            .fillna("")
            .astype(str)
            .str.strip()
        )


        promo_sales.loc[
            promo_sales["promo_display"] == "",
            "promo_display"
        ] = "No Promo Code"


        promo_group = (
            promo_sales
            .groupby(
                "promo_display"
            )
            .agg(
                Transaction_Count=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        promo_group["Profit Margin"] = (
            promo_group["Profit"]
            / promo_group["Revenue"]
            * 100
        ).where(
            promo_group["Revenue"] != 0
        ).fillna(0)


        promo_column_map = {
            "Transaction Count":
                "Transaction_Count",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin"
        }


        promo_column = promo_column_map[
            promo_metric
        ]


        promo_group = (
            promo_group
            .sort_values(
                promo_column,
                ascending=True
            )
            .tail(10)
        )


        positions = dynamic_bar_positions(
            promo_group[promo_column]
        )


        fig = go.Figure(
            go.Bar(
                x=promo_group[promo_column],
                y=promo_group["promo_display"],
                orientation="h",
                text=bar_text(
                    promo_group[promo_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(330)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "Customer Loyalty Performance",
            "Compare loyalty types by the selected metric"
        )


        loyalty_metric = st.selectbox(
            "Select the metric to compare loyalty types",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="loyalty_metric"
        )


        loyalty_group = (
            filtered_sales
            .groupby(
                "loyalty_tier"
            )
            .agg(
                Transactions=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        loyalty_group["Profit Margin"] = (
            loyalty_group["Profit"]
            / loyalty_group["Revenue"]
            * 100
        ).where(
            loyalty_group["Revenue"] != 0
        ).fillna(0)


        loyalty_column_map = {
            "Transaction Count":
                "Transactions",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin"
        }


        loyalty_column = loyalty_column_map[
            loyalty_metric
        ]


        loyalty_group = (
            loyalty_group
            .sort_values(
                loyalty_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            loyalty_group[loyalty_column]
        )


        fig = go.Figure(
            go.Bar(
                x=loyalty_group[loyalty_column],
                y=loyalty_group["loyalty_tier"],
                orientation="h",
                text=bar_text(
                    loyalty_group[loyalty_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(330)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # PAYMENT METHODS
    # =====================================================

    section_header(
        "Payment Methods"
    )


    payment_metric = st.selectbox(
        "Select the metric to compare payment methods",
        [
            "Transaction Count",
            "Total Revenue"
        ],
        key="payment_metric"
    )


    payment_group = (
        filtered_sales
        .groupby(
            "payment_method",
            dropna=False
        )
        .agg(
            Transaction_Count=(
                "sale_id",
                "nunique"
            ),
            Total_Revenue=(
                "total_amount",
                "sum"
            )
        )
        .reset_index()
    )


    payment_group["payment_method"] = (
        payment_group["payment_method"]
        .fillna("Unknown")
    )


    payment_column = (
        "Transaction_Count"
        if payment_metric == "Transaction Count"
        else "Total_Revenue"
    )


    payment_group = (
        payment_group
        .sort_values(
            payment_column,
            ascending=True
        )
    )


    positions = dynamic_bar_positions(
        payment_group[payment_column]
    )


    fig = go.Figure(
        go.Bar(
            x=payment_group[payment_column],
            y=payment_group["payment_method"],
            orientation="h",
            text=bar_text(
                payment_group[payment_column]
            ),
            textposition=positions,
            textfont=dict(
                size=12,
                color="#ffffff"
            ),
            cliponaxis=False
        )
    )


    fig.update_layout(
        **standard_bar_layout(330)
    )


    st.plotly_chart(
        fig,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )


    # =====================================================
    # DISCOUNT STATUS
    # =====================================================

    section_header(
        "Discount Status & Product Volume"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Discount Status",
            "Compare orders with discounts against orders marked No Discount"
        )


        discount_sales = (
            filtered_sales.copy()
        )


        discount_sales["discount_status"] = (
            discount_sales["discount_pct"] > 0
        ).map(
            {
                True: "With Discount",
                False: "No Discount"
            }
        )


        discount_group = (
            discount_sales
            .groupby(
                "discount_status"
            )
            .agg(
                Orders=(
                    "sale_id",
                    "nunique"
                )
            )
            .reset_index()
        )


        positions = dynamic_bar_positions(
            discount_group["Orders"]
        )


        fig = go.Figure(
            go.Bar(
                x=discount_group["discount_status"],
                y=discount_group["Orders"],
                text=bar_text(
                    discount_group["Orders"]
                ),
                textposition=positions,
                textfont=dict(
                    size=13,
                    color="#ffffff"
                )
            )
        )


        fig.update_layout(
            height=320,
            margin=dict(
                l=10,
                r=40,
                t=15,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False
            ),
            yaxis=dict(
                gridcolor="#243044"
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "Quantity by Category",
            "Compare total units sold across product categories"
        )


        quantity_group = (
            filtered_sales
            .groupby(
                "category_name"
            )["quantity"]
            .sum()
            .reset_index()
            .sort_values(
                "quantity",
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            quantity_group["quantity"]
        )


        fig = go.Figure(
            go.Bar(
                x=quantity_group["quantity"],
                y=quantity_group["category_name"],
                orientation="h",
                text=bar_text(
                    quantity_group["quantity"]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(320)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # TIME TREND
    # =====================================================

    section_header(
        "Time-Based Sales Trend"
    )


    trend_col1, trend_col2 = st.columns(2)


    with trend_col1:

        time_granularity = st.selectbox(
            "Time Granularity",
            [
                "Years",
                "Quarters",
                "Months"
            ],
            key="time_granularity"
        )


    with trend_col2:

        trend_metric = st.selectbox(
            "Select the metric to compare over time",
            [
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="trend_metric"
        )


    trend_sales = filtered_sales[
        filtered_sales["sale_date"]
        .notna()
    ].copy()


    if len(trend_sales) > 0:

        if time_granularity == "Years":

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.year
                .astype(str)
            )

        elif time_granularity == "Quarters":

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.to_period("Q")
                .astype(str)
            )

        else:

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.to_period("M")
                .astype(str)
            )


        trend_group = (
            trend_sales
            .groupby("period")
            .agg(
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        trend_group["Profit Margin"] = (
            trend_group["Profit"]
            / trend_group["Revenue"]
            * 100
        ).where(
            trend_group["Revenue"] != 0
        ).fillna(0)


        chart_title(
            f"{trend_metric} Trend",
            f"Compare {trend_metric.lower()} across {time_granularity.lower()}"
        )


        fig = go.Figure(
            go.Scatter(
                x=trend_group["period"],
                y=trend_group[trend_metric],
                mode="lines+markers",
                name=trend_metric,
                line=dict(
                    width=3
                ),
                marker=dict(
                    size=8
                ),
                hovertemplate=(
                    "%{x}"
                    "<br>"
                    + trend_metric
                    + ": %{y:,.2f}"
                    + "<extra></extra>"
                )
            )
        )


        fig.update_layout(
            height=380,
            margin=dict(
                l=10,
                r=20,
                t=15,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False,
                rangeslider=dict(
                    visible=False
                ),
                fixedrange=False
            ),
            yaxis=dict(
                gridcolor="#243044",
                fixedrange=False
            ),
            dragmode="zoom"
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False,
                "modeBarButtonsToAdd": [
                    "zoomIn2d",
                    "zoomOut2d",
                    "resetScale2d"
                ]
            }
        )


        missing_dates = (
            filtered_sales["sale_date"]
            .isna()
            .sum()
        )


        if missing_dates > 0:

            st.caption(
                f"{missing_dates:,.2f} records have no valid "
                "sale date and are excluded from the time trend."
            )


    else:

        st.info(
            "No valid sale dates are available "
            "for the selected filters."
        )


    # =====================================================
    # MARKET COVERAGE MAP
    # =====================================================

    section_header(
        "Market Coverage Map"
    )


    chart_title(
        "Customer Demand vs Existing Stores",
        "Compare where customers are located against existing pharmacy locations"
    )


    map_col1, map_col2 = st.columns(2)


    with map_col1:

        map_metric = st.selectbox(
            "Size customer demand by",
            [
                "Transaction Count",
                "Revenue",
                "Profit"
            ],
            key="market_map_metric"
        )


    with map_col2:

        demand_view = st.selectbox(
            "Customer demand view",
            [
                "All Customer Cities",
                "High Demand Cities"
            ],
            key="market_demand_view"
        )


    # =====================================================
    # STORE CITY AGGREGATION
    # =====================================================

    store_geo = (
        filtered_sales
        .groupby(
            "store_city",
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
        .rename(
            columns={
                "store_city": "city"
            }
        )
    )


    store_geo["Profit Margin"] = (
        store_geo["Profit"]
        / store_geo["Revenue"]
        * 100
    ).where(
        store_geo["Revenue"] != 0
    ).fillna(0)


    # =====================================================
    # CUSTOMER CITY AGGREGATION
    # =====================================================

    customer_geo = (
        filtered_sales
        .groupby(
            "customer_city",
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
        .rename(
            columns={
                "customer_city": "city"
            }
        )
    )


    customer_geo["Profit Margin"] = (
        customer_geo["Profit"]
        / customer_geo["Revenue"]
        * 100
    ).where(
        customer_geo["Revenue"] != 0
    ).fillna(0)


    # =====================================================
    # HIGH DEMAND FILTER
    # =====================================================

    if (
        demand_view == "High Demand Cities"
        and len(customer_geo) > 0
    ):

        demand_threshold = (
            customer_geo["Transactions"]
            .quantile(0.75)
        )

        customer_geo = customer_geo[
            customer_geo["Transactions"]
            >= demand_threshold
        ].copy()


    # =====================================================
    # COMBINED CITY LIST
    # =====================================================

    all_map_cities = sorted(
        set(
            store_geo["city"]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )
        |
        set(
            customer_geo["city"]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )
    )


    coordinates = geocode_cities(
        all_map_cities
    )


    # =====================================================
    # APPLY COORDINATES
    # =====================================================

    store_geo["lat"] = (
        store_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lat")
        )
    )


    store_geo["lon"] = (
        store_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lon")
        )
    )


    store_geo = store_geo.dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()


    customer_geo["lat"] = (
        customer_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lat")
        )
    )


    customer_geo["lon"] = (
        customer_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lon")
        )
    )


    customer_geo = customer_geo.dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()


    # =====================================================
    # COMBINED MAP
    # =====================================================

    map_points = pd.concat(
        [
            store_geo[
                [
                    "lat",
                    "lon"
                ]
            ],
            customer_geo[
                [
                    "lat",
                    "lon"
                ]
            ]
        ],
        ignore_index=True
    )


    if len(map_points) > 0:

        center_lat = (
            map_points["lat"]
            .mean()
        )

        center_lon = (
            map_points["lon"]
            .mean()
        )


        metric_column_map = {
            "Transaction Count":
                "Transactions",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit"
        }


        size_column = metric_column_map[
            map_metric
        ]


        fig = go.Figure()


        # =================================================
        # CUSTOMER DEMAND
        # =================================================

        if len(customer_geo) > 0:

            max_customer_value = (
                customer_geo[size_column]
                .max()
            )


            if max_customer_value > 0:

                customer_sizeref = (
                    2.0
                    * max_customer_value
                    / (32 ** 2)
                )

            else:

                customer_sizeref = 1


            fig.add_trace(
                go.Scattermap(
                    lat=customer_geo["lat"],
                    lon=customer_geo["lon"],
                    mode="markers",
                    name="Customer Demand",
                    hovertemplate=(
                        "<b>Customer Demand</b>"
                        "<br>City: %{customdata[0]}"
                        "<br>Transactions: %{customdata[1]:,.0f}"
                        "<br>Revenue: $%{customdata[2]:,.2f}"
                        "<br>Profit: $%{customdata[3]:,.2f}"
                        "<extra></extra>"
                    ),
                    customdata=customer_geo[
                        [
                            "city",
                            "Transactions",
                            "Revenue",
                            "Profit"
                        ]
                    ].values,
                    marker=dict(
                        size=customer_geo[
                            size_column
                        ],
                        sizemode="area",
                        sizeref=customer_sizeref,
                        sizemin=9,
                        symbol="circle",
                        color="#f472b6",
                        opacity=0.55,
                        allowoverlap=True
                    )
                )
            )


        # =================================================
        # STORE LOCATIONS
        # =================================================

        if len(store_geo) > 0:

            fig.add_trace(
                go.Scattermap(
                    lat=store_geo["lat"],
                    lon=store_geo["lon"],
                    mode="markers",
                    name="Existing Stores",
                    hovertemplate=(
                        "<b>Existing Store Location</b>"
                        "<br>City: %{customdata[0]}"
                        "<br>Transactions: %{customdata[1]:,.0f}"
                        "<br>Revenue: $%{customdata[2]:,.2f}"
                        "<br>Profit: $%{customdata[3]:,.2f}"
                        "<extra></extra>"
                    ),
                    customdata=store_geo[
                        [
                            "city",
                            "Transactions",
                            "Revenue",
                            "Profit"
                        ]
                    ].values,
                    marker=dict(
                        size=18,
                        symbol="pharmacy",
                        color="#22d3ee",
                        opacity=1
                    )
                )
            )


        # =================================================
        # MAP LAYOUT
        # =================================================

        fig.update_layout(
            height=560,
            margin=dict(
                l=0,
                r=0,
                t=0,
                b=0
            ),
            paper_bgcolor="#111827",
            map=dict(
                style="dark",
                center=dict(
                    lat=center_lat,
                    lon=center_lon
                ),
                zoom=4.5
            ),
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=0.015,
                xanchor="left",
                x=0.015,
                bgcolor="rgba(17,24,39,0.88)",
                bordercolor="#273449",
                borderwidth=1,
                font=dict(
                    color="#ffffff"
                )
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False
            }
        )


        st.caption(
            "Cyan pharmacy markers represent existing stores. "
            "Pink circles represent customer demand by city. "
            "Circle size represents the selected demand metric. "
            "Locations are city-level coordinates."
        )


    else:

        st.warning(
            "No valid store or customer city coordinates "
            "could be resolved."
        )


    # =====================================================
    # COVERAGE ANALYSIS
    # =====================================================

    section_header(
        "Market Coverage Analysis"
    )


    coverage_data = nearest_store_analysis(
        customer_geo=(
            (
                customer_geo
                if demand_view == "All Customer Cities"
                else (
                    (
                        filtered_sales
                        .groupby(
                            "customer_city",
                            dropna=False
                        )
                        .agg(
                            Transactions=(
                                "sale_id",
                                "nunique"
                            ),
                            Revenue=(
                                "total_amount",
                                "sum"
                            ),
                            Profit=(
                                "profit",
                                "sum"
                            )
                        )
                        .reset_index()
                        .rename(
                            columns={
                                "customer_city":
                                    "city"
                            }
                        )
                    )
                    .assign(
                        **{
                            "Profit Margin":
                            lambda x:
                            (
                                x["Profit"]
                                / x["Revenue"]
                                * 100
                            ).where(
                                x["Revenue"] != 0
                            ).fillna(0)
                        }
                    )
                )
            )
        ),
        store_geo=store_geo
    )


    if len(coverage_data) > 0:

        st.markdown(
            """
            <div style="
                color:#ffffff;
                font-size:12px;
                margin-bottom:12px;
            ">
                Customer cities are evaluated against their nearest
                existing store using city-level geographic distance.
            </div>
            """,
            unsafe_allow_html=True
        )


        coverage_col1, coverage_col2 = st.columns(2)


        with coverage_col1:

            distance_threshold = st.slider(
                "Minimum distance to consider underserved (km)",
                min_value=5,
                max_value=100,
                value=25,
                step=5,
                key="coverage_distance_threshold"
            )


        with coverage_col2:

            demand_threshold = st.slider(
                "Minimum customer transactions",
                min_value=1,
                max_value=max(
                    1,
                    int(
                        coverage_data[
                            "Customer Transactions"
                        ].max()
                    )
                ),
                value=min(
                    10,
                    max(
                        1,
                        int(
                            coverage_data[
                                "Customer Transactions"
                            ].max()
                        )
                    )
                ),
                step=1,
                key="coverage_demand_threshold"
            )


        coverage_data[
            "Potential Expansion"
        ] = (
            (
                coverage_data["Distance (km)"]
                >= distance_threshold
            )
            &
            (
                coverage_data["Customer Transactions"]
                >= demand_threshold
            )
        )


        expansion_data = (
            coverage_data[
                coverage_data[
                    "Potential Expansion"
                ]
            ]
            .copy()
            .sort_values(
                [
                    "Customer Transactions",
                    "Distance (km)"
                ],
                ascending=[
                    False,
                    False
                ]
            )
        )


        # =================================================
        # EXPANSION KPIs
        # =================================================

        k1, k2, k3 = st.columns(3)


        expansion_count = len(
            expansion_data
        )


        expansion_transactions = (
            expansion_data[
                "Customer Transactions"
            ].sum()
            if expansion_count > 0
            else 0
        )


        expansion_revenue = (
            expansion_data[
                "Customer Revenue"
            ].sum()
            if expansion_count > 0
            else 0
        )


        expansion_cards = [
            (
                "Potential Expansion Areas",
                f"{expansion_count:,.0f}",
                "Customer cities meeting both conditions"
            ),
            (
                "Demand in These Areas",
                f"{expansion_transactions:,.0f}",
                "Customer transactions"
            ),
            (
                "Associated Revenue",
                money(expansion_revenue),
                "Revenue generated from these cities"
            )
        ]


        for col, card in zip(
            [k1, k2, k3],
            expansion_cards
        ):

            with col:

                st.markdown(
                    f"""
                    <div class="kpi-card">
                        <div class="kpi-label">
                            {card[0]}
                        </div>
                        <div class="kpi-value">
                            {card[1]}
                        </div>
                        <div class="kpi-sub">
                            {card[2]}
                        </div>
                    </div>
                    """,
                    unsafe_allow_html=True
                )


        # =================================================
        # EXPANSION TABLE
        # =================================================

        chart_title(
            "Potential Expansion Areas",
            "High customer demand combined with greater distance from the nearest existing store"
        )


        if len(expansion_data) > 0:

            expansion_display = (
                expansion_data[
                    [
                        "Customer City",
                        "Customer Transactions",
                        "Customer Revenue",
                        "Customer Profit",
                        "Nearest Store",
                        "Distance (km)"
                    ]
                ]
                .copy()
            )


            expansion_display[
                "Customer Revenue"
            ] = expansion_display[
                "Customer Revenue"
            ].round(2)


            expansion_display[
                "Customer Profit"
            ] = expansion_display[
                "Customer Profit"
            ].round(2)


            expansion_display[
                "Distance (km)"
            ] = expansion_display[
                "Distance (km)"
            ].round(1)


            st.dataframe(
                expansion_display,
                use_container_width=True,
                hide_index=True
            )


        else:

            st.info(
                "No customer cities currently meet "
                "the selected expansion criteria."
            )


        # =================================================
        # NEAREST STORE ANALYSIS
        # =================================================

        chart_title(
            "Customer City Coverage",
            "Distance and activity of the nearest existing store"
        )


        coverage_display = (
            coverage_data[
                [
                    "Customer City",
                    "Customer Transactions",
                    "Customer Revenue",
                    "Nearest Store",
                    "Nearest Store Transactions",
                    "Distance (km)"
                ]
            ]
            .copy()
            .sort_values(
                "Distance (km)",
                ascending=False
            )
        )


        coverage_display[
            "Customer Revenue"
        ] = coverage_display[
            "Customer Revenue"
        ].round(2)


        coverage_display[
            "Distance (km)"
        ] = coverage_display[
            "Distance (km)"
        ].round(1)


        st.dataframe(
            coverage_display,
            use_container_width=True,
            hide_index=True
        )


    else:

        st.info(
            "Coverage analysis requires both customer "
            "and store city coordinates."
        )


    # =====================================================
    # STORE OVERLAP ANALYSIS
    # =====================================================

    section_header(
        "Store Proximity & Possible Overlap"
    )


    st.markdown(
        """
        <div style="
            color:#ffffff;
            font-size:12px;
            margin-bottom:12px;
        ">
            Identify stores located close to each other and compare
            their transaction activity. This is a city-level analysis
            when exact store coordinates are not available.
        </div>
        """,
        unsafe_allow_html=True
    )


    # =====================================================
    # STORE-LEVEL SALES
    # =====================================================

    store_details = (
        filtered_sales
        .groupby(
            [
                "store_id",
                "store_name",
                "store_city"
            ],
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
    )


    store_details = store_details.merge(
        store_geo[
            [
                "city",
                "lat",
                "lon"
            ]
        ],
        left_on="store_city",
        right_on="city",
        how="left"
    )


    store_details = store_details.drop(
        columns=[
            "city"
        ],
        errors="ignore"
    )


    overlap_data = store_overlap_analysis(
        store_details
    )


    if len(overlap_data) > 0:

        overlap_col1, overlap_col2 = st.columns(2)


        with overlap_col1:

            overlap_distance = st.slider(
                "Consider stores close within (km)",
                min_value=1,
                max_value=50,
                value=10,
                step=1,
                key="overlap_distance"
            )


        with overlap_col2:

            low_activity_threshold = st.slider(
                "Low transaction threshold per store",
                min_value=1,
                max_value=max(
                    1,
                    int(
                        store_details[
                            "Transactions"
                        ].max()
                    )
                ),
                value=min(
                    20,
                    max(
                        1,
                        int(
                            store_details[
                                "Transactions"
                            ].max()
                        )
                    )
                ),
                step=1,
                key="low_activity_threshold"
            )


        overlap_data["Potential Overlap"] = (
            (
                overlap_data[
                    "Distance (km)"
                ]
                <= overlap_distance
            )
            &
            (
                overlap_data[
                    "Transactions A"
                ]
                <= low_activity_threshold
            )
            &
            (
                overlap_data[
                    "Transactions B"
                ]
                <= low_activity_threshold
            )
        )


        potential_overlap = (
            overlap_data[
                overlap_data[
                    "Potential Overlap"
                ]
            ]
            .copy()
            .sort_values(
                [
                    "Distance (km)",
                    "Combined Transactions"
                ],
                ascending=[
                    True,
                    True
                ]
            )
        )


        overlap_k1, overlap_k2 = st.columns(2)


        with overlap_k1:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        Potential Store Overlaps
                    </div>
                    <div class="kpi-value">
                        {len(potential_overlap):,.0f}
                    </div>
                    <div class="kpi-sub">
                        Pairs meeting distance + activity conditions
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


        with overlap_k2:

            total_overlap_transactions = (
                potential_overlap[
                    "Combined Transactions"
                ].sum()
                if len(potential_overlap) > 0
                else 0
            )


            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        Transactions Across These Pairs
                    </div>
                    <div class="kpi-value">
                        {total_overlap_transactions:,.0f}
                    </div>
                    <div class="kpi-sub">
                        Combined activity of flagged stores
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


        chart_title(
            "Potential Store Overlap",
            "Nearby stores with relatively low transaction activity"
        )


        if len(potential_overlap) > 0:

            overlap_display = (
                potential_overlap[
                    [
                        "Store A",
                        "City A",
                        "Transactions A",
                        "Store B",
                        "City B",
                        "Transactions B",
                        "Distance (km)",
                        "Combined Transactions"
                    ]
                ]
                .copy()
            )


            overlap_display[
                "Distance (km)"
            ] = overlap_display[
                "Distance (km)"
            ].round(1)


            st.dataframe(
                overlap_display,
                use_container_width=True,
                hide_index=True
            )


        else:

            st.info(
                "No nearby store pairs currently meet "
                "the selected low-activity criteria."
            )


    else:

        st.info(
            "Store overlap analysis requires at least two "
            "stores with resolvable city coordinates."
        )



elif page == "Customer Loyalty":

    # =====================================================
    # PAGE HEADER
    # =====================================================

    st.markdown(
        '<div class="main-title">Customer Loyalty</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="main-subtitle">'
        'Customer Base · Geographic Distribution · Loyalty Performance'
        '</div>',
        unsafe_allow_html=True
    )


    # =====================================================
    # GENERAL FILTERS
    # =====================================================

    section_header(
        "Filters"
    )

    filter_col1, filter_col2 = st.columns(2)

    with filter_col1:
        loyalty_city_options = ["All Cities"] + (
            customers["city"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_loyalty_city = st.selectbox(
            "City",
            loyalty_city_options,
            key="customer_loyalty_filter_city"
        )

    with filter_col2:
        loyalty_tier_options = ["All Loyalty Tiers"] + (
            customers["loyalty_tier"]
            .fillna("No Loyalty")
            .astype(str)
            .str.strip()
            .replace("", "No Loyalty")
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_loyalty_tier = st.selectbox(
            "Loyalty Tier",
            loyalty_tier_options,
            key="customer_loyalty_filter_tier"
        )

    # ---------------------------------------------------------
    # APPLY GENERAL FILTERS
    # ---------------------------------------------------------

    if selected_loyalty_city != "All Cities":
        filtered_customer_ids = customers.loc[
            customers["city"].fillna("Unknown").astype(str).str.strip()
            == selected_loyalty_city,
            "customer_id"
        ].tolist()

        customers = customers[
            customers["customer_id"].isin(filtered_customer_ids)
        ]

        sales = sales[
            sales["customer_id"].isin(filtered_customer_ids)
        ]

    if selected_loyalty_tier != "All Loyalty Tiers":
        filtered_customer_ids = customers.loc[
            customers["loyalty_tier"].fillna("No Loyalty").astype(str).str.strip()
            == selected_loyalty_tier,
            "customer_id"
        ].tolist()

        customers = customers[
            customers["customer_id"].isin(filtered_customer_ids)
        ]

        sales = sales[
            sales["customer_id"].isin(filtered_customer_ids)
        ]

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    # =====================================================
    # CUSTOMER OVERVIEW
    # =====================================================

    section_header(
        "Customer Overview"
    )


    total_customers = (
        customers["customer_id"]
        .nunique()
    )


    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">
                Total Customers
            </div>
            <div class="kpi-value">
                {total_customers:,.0f}
            </div>
            <div class="kpi-sub">
                Unique customers in the customer base
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


    # =====================================================
    # CUSTOMER GEOGRAPHIC DISTRIBUTION
    # =====================================================

    section_header(
        "Customer Geographic Distribution"
    )


    left, right = st.columns(2)


    # =====================================================
    # CUSTOMER CITY DATA
    # =====================================================

    customer_city_group = (
        customers
        .copy()
    )


    customer_city_group["city"] = (
        customer_city_group["city"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )


    customer_city_group = (
        customer_city_group[
            customer_city_group["city"]
            != ""
        ]
    )


    customer_city_group = (
        customer_city_group
        .groupby(
            "city",
            dropna=False
        )
        .agg(
            Customers=(
                "customer_id",
                "nunique"
            )
        )
        .reset_index()
    )


    customer_city_group = (
        customer_city_group
        .sort_values(
            "Customers",
            ascending=False
        )
    )


    # =====================================================
    # GEOCODE CUSTOMER CITIES
    # =====================================================

    loyalty_cities = (
        customer_city_group["city"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )


    loyalty_coordinates = geocode_cities(
        loyalty_cities
    )


    customer_city_group["lat"] = (
        customer_city_group["city"]
        .map(
            lambda x:
            loyalty_coordinates.get(
                str(x).strip(),
                {}
            ).get("lat")
        )
    )


    customer_city_group["lon"] = (
        customer_city_group["city"]
        .map(
            lambda x:
            loyalty_coordinates.get(
                str(x).strip(),
                {}
            ).get("lon")
        )
    )


    customer_city_map = (
        customer_city_group
        .dropna(
            subset=[
                "lat",
                "lon"
            ]
        )
        .copy()
    )


    # =====================================================
    # CUSTOMER MAP
    # =====================================================

    with left:

        chart_title(
            "Customer Distribution by City",
            "Circle size represents the number of customers in each city"
        )


        if len(customer_city_map) > 0:

            max_customers = (
                customer_city_map["Customers"]
                .max()
            )


            if max_customers > 0:

                customer_sizeref = (
                    2.0
                    * max_customers
                    / (32 ** 2)
                )

            else:

                customer_sizeref = 1


            map_points = go.Figure(
                go.Scattermap(
                    lat=customer_city_map["lat"],
                    lon=customer_city_map["lon"],
                    mode="markers",
                    name="Customers",

                    customdata=customer_city_map[
                        [
                            "city",
                            "Customers"
                        ]
                    ].values,

                    hovertemplate=(
                        "<b>%{customdata[0]}</b>"
                        "<br>Customers: "
                        "%{customdata[1]:,.0f}"
                        "<extra></extra>"
                    ),

                    marker=dict(
                        size=customer_city_map[
                            "Customers"
                        ],
                        sizemode="area",
                        sizeref=customer_sizeref,
                        sizemin=9,
                        symbol="circle",
                        color="#f472b6",
                        opacity=0.65,
                        allowoverlap=True
                    )
                )
            )


            map_center_lat = (
                customer_city_map["lat"]
                .mean()
            )

            map_center_lon = (
                customer_city_map["lon"]
                .mean()
            )


            map_points.update_layout(
                height=430,
                margin=dict(
                    l=0,
                    r=0,
                    t=0,
                    b=0
                ),
                paper_bgcolor="#111827",
                map=dict(
                    style="dark",
                    center=dict(
                        lat=map_center_lat,
                        lon=map_center_lon
                    ),
                    zoom=4.5
                )
            )


            st.plotly_chart(
                map_points,
                use_container_width=True,
                config={
                    "displayModeBar": True,
                    "scrollZoom": True,
                    "displaylogo": False
                }
            )


        else:

            st.warning(
                "No customer city coordinates could be resolved."
            )


    # =====================================================
    # CUSTOMERS BY CITY BAR
    # =====================================================

    with right:

        chart_title(
            "Customers by City",
            "Compare the size of the customer base across cities"
        )


        city_bar = (
            customer_city_group
            .sort_values(
                "Customers",
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            city_bar["Customers"]
        )


        fig = go.Figure(
            go.Bar(
                x=city_bar["Customers"],
                y=city_bar["city"],
                orientation="h",

                text=bar_text(
                    city_bar["Customers"]
                ),

                textposition=positions,

                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),

                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(
                max(
                    430,
                    len(city_bar) * 28
                )
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # LOYALTY PERFORMANCE
    # =====================================================

    section_header(
        "Loyalty Performance"
    )


    chart_title(
        "Loyalty Tier Comparison",
        "Compare customer loyalty groups by the selected business metric"
    )


    loyalty_metric = st.selectbox(
        "Select the metric to compare loyalty tiers",
        [
            "Customers",
            "Transaction Count",
            "Revenue",
            "Profit",
            "Profit Margin"
        ],
        key="customer_loyalty_metric"
    )


    # =====================================================
    # CUSTOMER LOYALTY BASE
    # =====================================================

    loyalty_customers = customers.copy()


    loyalty_customers["loyalty_tier"] = (
        loyalty_customers["loyalty_tier"]
        .fillna("No Loyalty")
        .astype(str)
        .str.strip()
    )


    loyalty_customers.loc[
        loyalty_customers["loyalty_tier"] == "",
        "loyalty_tier"
    ] = "No Loyalty"


    loyalty_customer_group = (
        loyalty_customers
        .groupby(
            "loyalty_tier",
            dropna=False
        )
        .agg(
            Customers=(
                "customer_id",
                "nunique"
            )
        )
        .reset_index()
    )


    # =====================================================
    # SALES LOYALTY METRICS
    # =====================================================

    loyalty_sales = sales.copy()


    loyalty_sales["loyalty_tier"] = (
        loyalty_sales["loyalty_tier"]
        .fillna("No Loyalty")
        .astype(str)
        .str.strip()
    )


    loyalty_sales.loc[
        loyalty_sales["loyalty_tier"] == "",
        "loyalty_tier"
    ] = "No Loyalty"


    loyalty_sales_group = (
        loyalty_sales
        .groupby(
            "loyalty_tier",
            dropna=False
        )
        .agg(
            Transaction_Count=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
    )


    # =====================================================
    # COMBINE CUSTOMER + SALES METRICS
    # =====================================================

    loyalty_group = loyalty_customer_group.merge(
        loyalty_sales_group,
        on="loyalty_tier",
        how="outer"
    )


    loyalty_group[
        [
            "Customers",
            "Transaction_Count",
            "Revenue",
            "Profit"
        ]
    ] = loyalty_group[
        [
            "Customers",
            "Transaction_Count",
            "Revenue",
            "Profit"
        ]
    ].fillna(0)


    loyalty_group["Profit Margin"] = (
        loyalty_group["Profit"]
        / loyalty_group["Revenue"]
        * 100
    ).where(
        loyalty_group["Revenue"] != 0
    ).fillna(0)


    # =====================================================
    # METRIC SELECTION
    # =====================================================

    loyalty_metric_map = {
        "Customers":
            "Customers",

        "Transaction Count":
            "Transaction_Count",

        "Revenue":
            "Revenue",

        "Profit":
            "Profit",

        "Profit Margin":
            "Profit Margin"
    }


    loyalty_column = loyalty_metric_map[
        loyalty_metric
    ]


    loyalty_group = (
        loyalty_group
        .sort_values(
            loyalty_column,
            ascending=True
        )
    )


    # =====================================================
    # LOYALTY BAR CHART
    # =====================================================

    positions = dynamic_bar_positions(
        loyalty_group[loyalty_column]
    )


    fig = go.Figure(
        go.Bar(
            x=loyalty_group[
                loyalty_column
            ],

            y=loyalty_group[
                "loyalty_tier"
            ],

            orientation="h",

            text=bar_text(
                loyalty_group[
                    loyalty_column
                ]
            ),

            textposition=positions,

            textfont=dict(
                size=13,
                color="#ffffff"
            ),

            cliponaxis=False
        )
    )


    fig.update_layout(
        **standard_bar_layout(
            max(
                330,
                len(loyalty_group) * 65
            )
        )
    )


    st.plotly_chart(
        fig,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )


    # =====================================================
    # LOYALTY INTERPRETATION NOTE
    # =====================================================

    if loyalty_metric == "Customers":

        st.caption(
            "Customer count represents unique customers in each loyalty tier."
        )

    elif loyalty_metric == "Transaction Count":

        st.caption(
            "Transaction count represents unique sales transactions generated by each loyalty tier."
        )

    elif loyalty_metric == "Revenue":

        st.caption(
            "Revenue represents total sales revenue generated by each loyalty tier."
        )

    elif loyalty_metric == "Profit":

        st.caption(
            "Profit represents total profit generated by each loyalty tier."
        )

    else:

        st.caption(
            "Profit margin is calculated as total profit divided by total revenue for each loyalty tier."
        )
# =========================================================
    # CUSTOMER ACQUISITION TREND
    # =========================================================

    section_header(
        "Customer Acquisition"
    )

    trend_col1, trend_col2 = st.columns(2)

    with trend_col1:

        customer_granularity = st.selectbox(
            "Time Granularity",
            ["Years", "Quarters", "Months"],
            key="customer_acquisition_granularity"
        )

    with trend_col2:

        customer_metric = st.selectbox(
            "Select the metric to compare over time",
            [
                "New Customers",
                "Total Customers"
            ],
            key="customer_acquisition_metric"
        )


    customer_trend = customers.copy()

    customer_trend["signup_date"] = pd.to_datetime(
        customer_trend["signup_date"],
        errors="coerce"
    )

    customer_trend = customer_trend.dropna(
        subset=["signup_date"]
    )


    if len(customer_trend) > 0:

        # ---------------------------------------------------------
        # TIME GRANULARITY
        # ---------------------------------------------------------

        if customer_granularity == "Years":

            customer_trend["period"] = (
                customer_trend["signup_date"]
                .dt.year
                .astype(str)
            )

        elif customer_granularity == "Quarters":

            customer_trend["period"] = (
                customer_trend["signup_date"]
                .dt.to_period("Q")
                .astype(str)
            )

        else:

            customer_trend["period"] = (
                customer_trend["signup_date"]
                .dt.to_period("M")
                .astype(str)
            )


        # ---------------------------------------------------------
        # CUSTOMER COUNTS
        # ---------------------------------------------------------

        trend_group = (
            customer_trend
            .groupby("period")
            .size()
            .reset_index(name="New_Customers")
        )

        trend_group["Total_Customers"] = (
            trend_group["New_Customers"].cumsum()
        )


        # ---------------------------------------------------------
        # CHART
        # ---------------------------------------------------------

        chart_title(
            f"{customer_metric} Trend",
            f"Track {customer_metric.lower()} across "
            f"{customer_granularity.lower()}"
        )


        fig = go.Figure(
            go.Scatter(
                x=trend_group["period"],
                y=trend_group[
                    customer_metric.replace(" ", "_")
                ],
                mode="lines+markers",
                name=customer_metric,

                line=dict(
                    width=3,
                    color="#38BDF8"
                ),

                marker=dict(
                    size=8,
                    color="#38BDF8"
                ),

                hovertemplate=(
                    "%{x}<br>"
                    + customer_metric
                    + ": %{y:,.0f}"
                    + "<extra></extra>"
                )
            )
        )


        fig.update_layout(
            height=380,

            margin=dict(
                l=10,
                r=20,
                t=15,
                b=15
            ),

            paper_bgcolor="#111827",
            plot_bgcolor="#111827",

            font=dict(
                color="#e2e8f0"
            ),

            xaxis=dict(
                showgrid=False,
                rangeslider=dict(
                    visible=False
                ),
                fixedrange=False
            ),

            yaxis=dict(
                gridcolor="#243044",
                fixedrange=False
            ),

            dragmode="zoom"
        )


        st.plotly_chart(
            fig,
            use_container_width=True,

            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False,

                "modeBarButtonsToAdd": [
                    "zoomIn2d",
                    "zoomOut2d",
                    "resetScale2d"
                ]
            }
        )


    else:

        st.info(
            "No valid signup dates are available "
            "for the customer acquisition trend."
        )



elif page == "Inventory":


    # =========================================================
    # INVENTORY DATA PREPARATION
    # =========================================================

    inventory = stock.copy()
    product_catalog = products.copy()

    inventory["quantity"] = pd.to_numeric(
        inventory["quantity"],
        errors="coerce"
    ).fillna(0)

    inventory["reorder_level"] = pd.to_numeric(
        inventory["reorder_level"],
        errors="coerce"
    ).fillna(0)

    inventory["received_date"] = pd.to_datetime(
        inventory["received_date"],
        errors="coerce"
    )

    inventory["last_restocked"] = pd.to_datetime(
        inventory["last_restocked"],
        errors="coerce"
    )

    inventory["expiry_date"] = pd.to_datetime(
        inventory["expiry_date"],
        errors="coerce"
    )

    product_catalog["cost_price"] = pd.to_numeric(
        product_catalog["cost_price"],
        errors="coerce"
    ).fillna(0)


    # =========================================================
    # GENERAL FILTERS
    # =========================================================

    section_header(
        "Filters"
    )

    filter_col1, filter_col2, filter_col3, filter_col4 = st.columns(4)

    with filter_col1:
        store_options = ["All Stores"] + (
            stores["store_name"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_inventory_store = st.selectbox(
            "Store",
            store_options,
            key="inventory_filter_store"
        )

    with filter_col2:
        category_options = ["All Categories"] + (
            categories["category_name"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_inventory_category = st.selectbox(
            "Category",
            category_options,
            key="inventory_filter_category"
        )

    with filter_col3:
        supplier_options = ["All Suppliers"] + (
            suppliers["supplier_name"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_inventory_supplier = st.selectbox(
            "Supplier",
            supplier_options,
            key="inventory_filter_supplier"
        )

    # Product options depend on the selected Category and Supplier
    product_filter_catalog = product_catalog.copy()

    if selected_inventory_category != "All Categories":
        selected_category_ids = categories.loc[
            categories["category_name"].astype(str) == selected_inventory_category,
            "category_id"
        ].tolist()

        product_filter_catalog = product_filter_catalog[
            product_filter_catalog["category_id"].isin(selected_category_ids)
        ]

    if selected_inventory_supplier != "All Suppliers":
        selected_supplier_ids = suppliers.loc[
            suppliers["supplier_name"].astype(str) == selected_inventory_supplier,
            "supplier_id"
        ].tolist()

        product_filter_catalog = product_filter_catalog[
            product_filter_catalog["supplier_id"].isin(selected_supplier_ids)
        ]

    with filter_col4:
        product_options = ["All Products"] + (
            product_filter_catalog["product_name"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        selected_inventory_product = st.selectbox(
            "Product",
            product_options,
            key="inventory_filter_product"
        )

    # ---------------------------------------------------------
    # APPLY FILTERS
    # ---------------------------------------------------------

    if selected_inventory_store != "All Stores":
        selected_store_ids = stores.loc[
            stores["store_name"].astype(str) == selected_inventory_store,
            "store_id"
        ].tolist()

        inventory = inventory[
            inventory["store_id"].isin(selected_store_ids)
        ]

    if selected_inventory_category != "All Categories":
        selected_category_ids = categories.loc[
            categories["category_name"].astype(str) == selected_inventory_category,
            "category_id"
        ].tolist()

        selected_product_ids = product_catalog.loc[
            product_catalog["category_id"].isin(selected_category_ids),
            "product_id"
        ].tolist()

        inventory = inventory[
            inventory["product_id"].isin(selected_product_ids)
        ]

        product_catalog = product_catalog[
            product_catalog["category_id"].isin(selected_category_ids)
        ]

    if selected_inventory_supplier != "All Suppliers":
        selected_supplier_ids = suppliers.loc[
            suppliers["supplier_name"].astype(str) == selected_inventory_supplier,
            "supplier_id"
        ].tolist()

        selected_product_ids = product_catalog.loc[
            product_catalog["supplier_id"].isin(selected_supplier_ids),
            "product_id"
        ].tolist()

        inventory = inventory[
            inventory["product_id"].isin(selected_product_ids)
        ]

        product_catalog = product_catalog[
            product_catalog["supplier_id"].isin(selected_supplier_ids)
        ]

    if selected_inventory_product != "All Products":
        selected_product_ids = product_catalog.loc[
            product_catalog["product_name"].astype(str) == selected_inventory_product,
            "product_id"
        ].tolist()

        inventory = inventory[
            inventory["product_id"].isin(selected_product_ids)
        ]

        product_catalog = product_catalog[
            product_catalog["product_id"].isin(selected_product_ids)
        ]

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # EXCEL EXPORT HELPER
    # =========================================================

    def to_excel_bytes(dataframe):

        output = io.BytesIO()

        with pd.ExcelWriter(
            output,
            engine="openpyxl"
        ) as writer:

            dataframe.to_excel(
                writer,
                index=False,
                sheet_name="Inventory"
            )

        return output.getvalue()


    # =========================================================
    # MERGE PRODUCT INFORMATION
    # =========================================================

    inventory = inventory.merge(
        product_catalog[
            [
                "product_id",
                "product_name",
                "cost_price"
            ]
        ],
        on="product_id",
        how="left"
    )

    inventory["cost_price"] = (
        inventory["cost_price"]
        .fillna(0)
    )

    inventory["stock_value"] = (
        inventory["quantity"]
        * inventory["cost_price"]
    )


    # =========================================================
    # CURRENT STOCK BY PRODUCT
    # =========================================================

    current_stock = (
        inventory
        .groupby(
            [
                "product_id",
                "product_name",
                "cost_price"
            ],
            as_index=False
        )
        .agg(
            quantity=("quantity", "sum")
        )
    )

    current_stock["stock_value"] = (
        current_stock["quantity"]
        * current_stock["cost_price"]
    )


    # =========================================================
    # INVENTORY OVERVIEW
    # =========================================================

    section_header(
        "Inventory Overview"
    )


    supply_batches = (
        inventory["batch_number"]
        .dropna()
        .nunique()
    )

    total_stock_quantity = (
        current_stock["quantity"]
        .sum()
    )

    total_stock_value = (
        current_stock["stock_value"]
        .sum()
    )


    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Supply Batches
                </div>
                <div class="kpi-value">
                    {number(supply_batches)}
                </div>
                <div class="kpi-sub">
                    Total restocking batches
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Stock Quantity
                </div>
                <div class="kpi-value">
                    {number(total_stock_quantity)}
                </div>
                <div class="kpi-sub">
                    Units currently in stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Stock Value
                </div>
                <div class="kpi-value">
                    ${number(total_stock_value)}
                </div>
                <div class="kpi-sub">
                    Current inventory cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # =========================================================
    # SUPPLY MOVEMENT
    # =========================================================

    section_header(
        "Time-Based Supply Trend"
    )


    trend_col1, trend_col2 = st.columns(2)


    with trend_col1:

        supply_granularity = st.selectbox(
            "Time Granularity",
            [
                "Years",
                "Quarters",
                "Months"
            ],
            key="inventory_supply_granularity"
        )


    with trend_col2:

        supply_metric = st.selectbox(
            "Select the metric to compare over time",
            [
                "Supply Batches",
                "Supplied Quantity",
                "Supply Cost"
            ],
            key="inventory_supply_metric"
        )


    trend_supply = inventory[
        inventory["received_date"].notna()
    ].copy()


    if len(trend_supply) > 0:

        # -----------------------------------------------------
        # TIME GRANULARITY
        # -----------------------------------------------------

        if supply_granularity == "Years":

            trend_supply["period"] = (
                trend_supply["received_date"]
                .dt.year
                .astype(str)
            )

        elif supply_granularity == "Quarters":

            trend_supply["period"] = (
                trend_supply["received_date"]
                .dt.to_period("Q")
                .astype(str)
            )

        else:

            trend_supply["period"] = (
                trend_supply["received_date"]
                .dt.to_period("M")
                .astype(str)
            )


        # -----------------------------------------------------
        # SUPPLY COST
        # -----------------------------------------------------

        trend_supply["supply_cost"] = (
            trend_supply["quantity"]
            * trend_supply["cost_price"]
        )


        # -----------------------------------------------------
        # AGGREGATION
        # -----------------------------------------------------

        trend_group = (
            trend_supply
            .groupby("period")
            .agg(
                Supply_Batches=(
                    "batch_number",
                    "nunique"
                ),
                Supplied_Quantity=(
                    "quantity",
                    "sum"
                ),
                Supply_Cost=(
                    "supply_cost",
                    "sum"
                )
            )
            .reset_index()
        )


        trend_group["Supply Batches"] = (
            trend_group["Supply_Batches"]
        )

        trend_group["Supplied Quantity"] = (
            trend_group["Supplied_Quantity"]
        )

        trend_group["Supply Cost"] = (
            trend_group["Supply_Cost"]
        )


        # -----------------------------------------------------
        # CHART TITLE
        # -----------------------------------------------------

        chart_title(
            f"{supply_metric} Trend",
            (
                f"Compare {supply_metric.lower()} "
                f"across "
                f"{supply_granularity.lower()}"
            )
        )


        # -----------------------------------------------------
        # LINE CHART
        # -----------------------------------------------------

        fig = go.Figure(
            go.Scatter(
                x=trend_group["period"],
                y=trend_group[supply_metric],
                mode="lines+markers",
                name=supply_metric,

                line=dict(
                    width=3,
                    color="#38BDF8"
                ),

                marker=dict(
                    size=8,
                    color="#38BDF8"
                ),

                hovertemplate=(
                    "%{x}"
                    "<br>"
                    + supply_metric
                    + ": %{y:,.2f}"
                    + "<extra></extra>"
                )
            )
        )


        fig.update_layout(
            height=380,

            margin=dict(
                l=10,
                r=20,
                t=15,
                b=15
            ),

            paper_bgcolor="#111827",
            plot_bgcolor="#111827",

            font=dict(
                color="#e2e8f0"
            ),

            xaxis=dict(
                showgrid=False,
                rangeslider=dict(
                    visible=False
                ),
                fixedrange=False
            ),

            yaxis=dict(
                gridcolor="#243044",
                fixedrange=False
            ),

            dragmode="zoom"
        )


        st.plotly_chart(
            fig,
            use_container_width=True,

            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False,

                "modeBarButtonsToAdd": [
                    "zoomIn2d",
                    "zoomOut2d",
                    "resetScale2d"
                ]
            }
        )


        # -----------------------------------------------------
        # MISSING RECEIVED DATES
        # -----------------------------------------------------

        missing_received_dates = (
            inventory["received_date"]
            .isna()
            .sum()
        )


        if missing_received_dates > 0:

            st.caption(
                f"{missing_received_dates:,.2f} records have no valid "
                "received date and are excluded from the time trend."
            )


    else:

        st.info(
            "No valid received dates are available "
            "for the supply trend."
        )


    # =========================================================
    # PRODUCT AVAILABILITY
    # =========================================================

    section_header(
        "Product Availability"
    )


    catalog_products = (
        product_catalog["product_id"]
        .nunique()
    )


    products_in_stock = (
        current_stock.loc[
            current_stock["quantity"] > 0,
            "product_id"
        ]
        .nunique()
    )


    missing_products_count = max(
        catalog_products - products_in_stock,
        0
    )


    missing_products_pct = (
        missing_products_count
        / catalog_products
        * 100
        if catalog_products > 0
        else 0
    )


# ---------------------------------------------------------
    # AVAILABILITY CARDS
    # ---------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Catalog Products
                </div>
                <div class="kpi-value">
                    {number(catalog_products)}
                </div>
                <div class="kpi-sub">
                    Products registered in catalog
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Products in Stock
                </div>
                <div class="kpi-value">
                    {number(products_in_stock)}
                </div>
                <div class="kpi-sub">
                    Products currently available
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Missing Products
                </div>
                <div class="kpi-value">
                    {number(missing_products_count)}
                </div>
                <div class="kpi-sub">
                    Catalog products not currently stocked
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col4:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Missing Products %
                </div>
                <div class="kpi-value">
                    {missing_products_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total catalog products
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ---------------------------------------------------------
    # MISSING PRODUCTS DATA
    # ---------------------------------------------------------

    catalog_ids = set(
        product_catalog["product_id"]
    )


    stock_ids = set(
        current_stock.loc[
            current_stock["quantity"] > 0,
            "product_id"
        ]
    )


    missing_ids = (
        catalog_ids - stock_ids
    )


    missing_products = (
        product_catalog[
            product_catalog["product_id"].isin(
                missing_ids
            )
        ]
        [
            [
                "product_id",
                "product_name",
                "category_id",
                "brand"
            ]
        ]
        .sort_values("product_name")
    )


    # ---------------------------------------------------------
    # MISSING PRODUCTS EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export Missing Products",
        data=to_excel_bytes(
            missing_products
        ),
        file_name="missing_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_missing_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )

    # =========================================================
    # REORDER ALERTS
    # =========================================================

    section_header(
        "Reorder Alerts"
    )


    reorder_stock = (
        inventory
        .groupby(
            [
                "product_id",
                "product_name"
            ],
            as_index=False
        )
        .agg(
            current_quantity=(
                "quantity",
                "sum"
            ),
            reorder_level=(
                "reorder_level",
                "max"
            )
        )
    )


    reorder_products = (
        reorder_stock[
            reorder_stock["current_quantity"]
            <= reorder_stock["reorder_level"]
        ]
        .copy()
    )


    reorder_count = (
        reorder_products["product_id"]
        .nunique()
    )


    # ---------------------------------------------------------
    # REORDER KPI
    # ---------------------------------------------------------

    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">
                Products at Reorder Level
            </div>
            <div class="kpi-value">
                {number(reorder_count)}
            </div>
            <div class="kpi-sub">
                Products requiring restocking
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


    reorder_export = (
        reorder_products[
            [
                "product_id",
                "product_name",
                "current_quantity",
                "reorder_level"
            ]
        ]
        .sort_values("current_quantity")
    )


    # ---------------------------------------------------------
    # REORDER EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export Reorder Products",
        data=to_excel_bytes(
            reorder_export
        ),
        file_name="reorder_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_reorder_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # EXPIRY MONITORING
    # =========================================================

    section_header(
        "Expiry Monitoring"
    )


    today = (
        pd.Timestamp.today()
        .normalize()
    )


    risk_date = (
        today
        + pd.DateOffset(months=3)
    )


    # =========================================================
    # EXPIRED PRODUCTS
    # =========================================================

    expired_stock = inventory[
        inventory["expiry_date"].notna()
        &
        (
            inventory["expiry_date"]
            < today
        )
        &
        (
            inventory["quantity"] > 0
        )
    ].copy()


    expired_products_count = (
        expired_stock["product_id"]
        .nunique()
    )


    expired_quantity = (
        expired_stock["quantity"]
        .sum()
    )


    expired_cost = (
        expired_stock["stock_value"]
        .sum()
    )


    expired_quantity_pct = (
        expired_quantity
        / total_stock_quantity
        * 100
        if total_stock_quantity > 0
        else 0
    )


    expired_cost_pct = (
        expired_cost
        / total_stock_value
        * 100
        if total_stock_value > 0
        else 0
    )


    # ---------------------------------------------------------
    # EXPIRED MAIN CARDS
    # ---------------------------------------------------------

    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Products
                </div>
                <div class="kpi-value">
                    {number(expired_products_count)}
                </div>
                <div class="kpi-sub">
                    Products with expired stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Quantity
                </div>
                <div class="kpi-value">
                    {number(expired_quantity)}
                </div>
                <div class="kpi-sub">
                    Units already expired
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Stock Cost
                </div>
                <div class="kpi-value">
                    ${number(expired_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of expired inventory
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ---------------------------------------------------------
    # EXPIRED PERCENTAGE CARDS
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 14px;'></div>",
        unsafe_allow_html=True
    )


    col1, col2 = st.columns(2)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Quantity %
                </div>
                <div class="kpi-value">
                    {expired_quantity_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock quantity
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Expired Cost %
                </div>
                <div class="kpi-value">
                    {expired_cost_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    expired_export = (
        expired_stock[
            [
                "product_id",
                "product_name",
                "expiry_date",
                "quantity",
                "cost_price",
                "stock_value"
            ]
        ]
        .copy()
        .rename(
            columns={
                "stock_value": "total_cost"
            }
        )
    )


    # ---------------------------------------------------------
    # EXPIRED EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export Expired Products",
        data=to_excel_bytes(
            expired_export
        ),
        file_name="expired_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_expired_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # RECEIVED ALREADY EXPIRED
    # =========================================================

    section_header(
        "Received Already Expired"
    )


    received_expired = inventory[
        inventory["expiry_date"].notna()
        &
        inventory["received_date"].notna()
        &
        (
            inventory["received_date"]
            > inventory["expiry_date"]
        )
        &
        (
            inventory["quantity"] > 0
        )
    ].copy()


    received_expired_products_count = (
        received_expired["product_id"]
        .nunique()
    )


    received_expired_quantity = (
        received_expired["quantity"]
        .sum()
    )


    received_expired_cost = (
        received_expired["stock_value"]
        .sum()
    )


    received_expired_quantity_pct = (
        received_expired_quantity
        / total_stock_quantity
        * 100
        if total_stock_quantity > 0
        else 0
    )


    received_expired_cost_pct = (
        received_expired_cost
        / total_stock_value
        * 100
        if total_stock_value > 0
        else 0
    )


    # ---------------------------------------------------------
    # RECEIVED EXPIRED MAIN CARDS
    # ---------------------------------------------------------

    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received Expired Products
                </div>
                <div class="kpi-value">
                    {number(received_expired_products_count)}
                </div>
                <div class="kpi-sub">
                    Products received after expiry
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received Expired Quantity
                </div>
                <div class="kpi-value">
                    {number(received_expired_quantity)}
                </div>
                <div class="kpi-sub">
                    Units received after expiry
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received Expired Cost
                </div>
                <div class="kpi-value">
                    ${number(received_expired_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of affected inventory
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ---------------------------------------------------------
    # RECEIVED EXPIRED PERCENTAGE CARDS
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 14px;'></div>",
        unsafe_allow_html=True
    )


    col1, col2 = st.columns(2)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received Expired Quantity %
                </div>
                <div class="kpi-value">
                    {received_expired_quantity_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock quantity
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received Expired Cost %
                </div>
                <div class="kpi-value">
                    {received_expired_cost_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    received_expired_export = (
        received_expired[
            [
                "product_id",
                "product_name",
                "received_date",
                "expiry_date",
                "quantity",
                "cost_price",
                "stock_value"
            ]
        ]
        .copy()
        .rename(
            columns={
                "stock_value": "total_cost"
            }
        )
    )


    # ---------------------------------------------------------
    # RECEIVED EXPIRED EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export Received Expired Products",
        data=to_excel_bytes(
            received_expired_export
        ),
        file_name="received_expired_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_received_expired_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # AT-RISK PRODUCTS
    # =========================================================

    section_header(
        "At-Risk Products"
    )


    at_risk_stock = inventory[
        inventory["expiry_date"].notna()
        &
        (
            inventory["expiry_date"]
            >= today
        )
        &
        (
            inventory["expiry_date"]
            <= risk_date
        )
        &
        (
            inventory["quantity"] > 0
        )
    ].copy()


    at_risk_stock["days_remaining"] = (
        at_risk_stock["expiry_date"]
        - today
    ).dt.days


    at_risk_products_count = (
        at_risk_stock["product_id"]
        .nunique()
    )


    at_risk_quantity = (
        at_risk_stock["quantity"]
        .sum()
    )


    at_risk_cost = (
        at_risk_stock["stock_value"]
        .sum()
    )


    at_risk_quantity_pct = (
        at_risk_quantity
        / total_stock_quantity
        * 100
        if total_stock_quantity > 0
        else 0
    )


    at_risk_cost_pct = (
        at_risk_cost
        / total_stock_value
        * 100
        if total_stock_value > 0
        else 0
    )


    # ---------------------------------------------------------
    # AT-RISK MAIN CARDS
    # ---------------------------------------------------------

    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Products
                </div>
                <div class="kpi-value">
                    {number(at_risk_products_count)}
                </div>
                <div class="kpi-sub">
                    Expiring within 3 months
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Quantity
                </div>
                <div class="kpi-value">
                    {number(at_risk_quantity)}
                </div>
                <div class="kpi-sub">
                    Units expiring within 3 months
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Stock Cost
                </div>
                <div class="kpi-value">
                    ${number(at_risk_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of at-risk inventory
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ---------------------------------------------------------
    # AT-RISK PERCENTAGE CARDS
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 14px;'></div>",
        unsafe_allow_html=True
    )


    col1, col2 = st.columns(2)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Quantity %
                </div>
                <div class="kpi-value">
                    {at_risk_quantity_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock quantity
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    At-Risk Cost %
                </div>
                <div class="kpi-value">
                    {at_risk_cost_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    at_risk_export = (
        at_risk_stock[
            [
                "product_id",
                "product_name",
                "expiry_date",
                "days_remaining",
                "quantity",
                "cost_price",
                "stock_value"
            ]
        ]
        .copy()
        .rename(
            columns={
                "stock_value": "total_cost"
            }
        )
    )


    # ---------------------------------------------------------
    # AT-RISK EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export At-Risk Products",
        data=to_excel_bytes(
            at_risk_export
        ),
        file_name="at_risk_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_at_risk_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # RECEIVED ALREADY AT-RISK
    # =========================================================

    section_header(
        "Received Already At-Risk"
    )


    received_at_risk = inventory[
        inventory["expiry_date"].notna()
        &
        inventory["received_date"].notna()
        &
        (
            inventory["received_date"]
            >= (
                inventory["expiry_date"]
                - pd.DateOffset(months=3)
            )
        )
        &
        (
            inventory["received_date"]
            <= inventory["expiry_date"]
        )
        &
        (
            inventory["quantity"] > 0
        )
    ].copy()


    received_at_risk["days_remaining_at_receipt"] = (
        received_at_risk["expiry_date"]
        - received_at_risk["received_date"]
    ).dt.days


    received_at_risk_products_count = (
        received_at_risk["product_id"]
        .nunique()
    )


    received_at_risk_quantity = (
        received_at_risk["quantity"]
        .sum()
    )


    received_at_risk_cost = (
        received_at_risk["stock_value"]
        .sum()
    )


    received_at_risk_quantity_pct = (
        received_at_risk_quantity
        / total_stock_quantity
        * 100
        if total_stock_quantity > 0
        else 0
    )


    received_at_risk_cost_pct = (
        received_at_risk_cost
        / total_stock_value
        * 100
        if total_stock_value > 0
        else 0
    )


    # ---------------------------------------------------------
    # RECEIVED AT-RISK MAIN CARDS
    # ---------------------------------------------------------

    col1, col2, col3 = st.columns(3)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received At-Risk Products
                </div>
                <div class="kpi-value">
                    {number(received_at_risk_products_count)}
                </div>
                <div class="kpi-sub">
                    Received within final 3 months
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received At-Risk Quantity
                </div>
                <div class="kpi-value">
                    {number(received_at_risk_quantity)}
                </div>
                <div class="kpi-sub">
                    Units received within final 3 months
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received At-Risk Cost
                </div>
                <div class="kpi-value">
                    ${number(received_at_risk_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of affected inventory
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ---------------------------------------------------------
    # RECEIVED AT-RISK PERCENTAGE CARDS
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 14px;'></div>",
        unsafe_allow_html=True
    )


    col1, col2 = st.columns(2)


    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received At-Risk Quantity %
                </div>
                <div class="kpi-value">
                    {received_at_risk_quantity_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock quantity
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Received At-Risk Cost %
                </div>
                <div class="kpi-value">
                    {received_at_risk_cost_pct:.2f}%
                </div>
                <div class="kpi-sub">
                    Of total stock cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    received_at_risk_export = (
        received_at_risk[
            [
                "product_id",
                "product_name",
                "received_date",
                "expiry_date",
                "days_remaining_at_receipt",
                "quantity",
                "cost_price",
                "stock_value"
            ]
        ]
        .copy()
        .rename(
            columns={
                "stock_value": "total_cost"
            }
        )
    )


    # ---------------------------------------------------------
    # RECEIVED AT-RISK EXPORT
    # ---------------------------------------------------------

    st.markdown(
        "<div style='height: 18px;'></div>",
        unsafe_allow_html=True
    )


    st.download_button(
        label="Export Received At-Risk Products",
        data=to_excel_bytes(
            received_at_risk_export
        ),
        file_name="received_at_risk_products.xlsx",
        mime=(
            "application/vnd.openxmlformats-"
            "officedocument.spreadsheetml.sheet"
        ),
        key="export_received_at_risk_products"
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )
# =========================================================
    # INVENTORY RISK OVERVIEW
    # =========================================================

    section_header(
        "Inventory Risk Overview"
    )

    risk_metric = st.selectbox(
        "Select Risk Metric",
        [
            "Quantity",
            "Cost"
        ],
        key="inventory_risk_metric"
    )

    risk_chart_col1, risk_chart_col2 = st.columns(2)


    # =========================================================
    # PREPARE METRIC VALUES
    # =========================================================

    if risk_metric == "Quantity":

        current_total_value = total_stock_quantity
        expired_value = expired_quantity
        at_risk_value = at_risk_quantity

        received_expired_value = received_expired_quantity
        received_at_risk_value = received_at_risk_quantity

        metric_label = "Quantity"

    else:

        current_total_value = total_stock_value
        expired_value = expired_cost
        at_risk_value = at_risk_cost

        received_expired_value = received_expired_cost
        received_at_risk_value = received_at_risk_cost

        metric_label = "Cost"


    # =========================================================
    # CHART 1 — CURRENT INVENTORY RISK
    # =========================================================

    with risk_chart_col1:

        chart_title(
            "Current Inventory Risk",
            "Current stock compared with expired and at-risk inventory"
        )

        current_risk_data = pd.DataFrame({
            "Status": [
                "Total Stock",
                "Expired",
                "At-Risk"
            ],
            "Value": [
                current_total_value,
                expired_value,
                at_risk_value
            ]
        })

        fig_current_risk = go.Figure()

        fig_current_risk.add_trace(
            go.Bar(
                x=current_risk_data["Status"],
                y=current_risk_data["Value"],
                text=current_risk_data["Value"].apply(
                    lambda x: (
                        f"{x:,.0f}"
                        if risk_metric == "Quantity"
                        else f"${x:,.2f}"
                    )
                ),
                textposition="outside",
                marker=dict(
                    color=[
                        "#38BDF8",
                        "#EF4444",
                        "#F59E0B"
                    ]
                ),
                hovertemplate=(
                    "%{x}<br>"
                    + metric_label
                    + ": "
                    + (
                        "%{y:,.0f}"
                        if risk_metric == "Quantity"
                        else "$%{y:,.2f}"
                    )
                    + "<extra></extra>"
                )
            )
        )

        fig_current_risk.update_layout(
            height=380,
            margin=dict(
                l=10,
                r=20,
                t=25,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False
            ),
            yaxis=dict(
                gridcolor="#243044",
                title=metric_label
            ),
            showlegend=False
        )

        st.plotly_chart(
            fig_current_risk,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "displaylogo": False
            }
        )


    # =========================================================
    # CHART 2 — RECEIVED INVENTORY RISK
    # =========================================================

    with risk_chart_col2:

        chart_title(
            "Received Inventory Risk",
            "Inventory received expired or within its final 3 months"
        )

        received_risk_data = pd.DataFrame({
            "Status": [
                "Expired",
                "Received Expired",
                "At-Risk",
                "Received At-Risk"
            ],
            "Value": [
                expired_value,
                received_expired_value,
                at_risk_value,
                received_at_risk_value
            ]
        })

        fig_received_risk = go.Figure()

        fig_received_risk.add_trace(
            go.Bar(
                x=received_risk_data["Status"],
                y=received_risk_data["Value"],
                text=received_risk_data["Value"].apply(
                    lambda x: (
                        f"{x:,.0f}"
                        if risk_metric == "Quantity"
                        else f"${x:,.2f}"
                    )
                ),
                textposition="outside",
                marker=dict(
                    color=[
                        "#EF4444",
                        "#B91C1C",
                        "#F59E0B",
                        "#D97706"
                    ]
                ),
                hovertemplate=(
                    "%{x}<br>"
                    + metric_label
                    + ": "
                    + (
                        "%{y:,.0f}"
                        if risk_metric == "Quantity"
                        else "$%{y:,.2f}"
                    )
                    + "<extra></extra>"
                )
            )
        )

        fig_received_risk.update_layout(
            height=380,
            margin=dict(
                l=10,
                r=20,
                t=25,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False
            ),
            yaxis=dict(
                gridcolor="#243044",
                title=metric_label
            ),
            showlegend=False
        )

        st.plotly_chart(
            fig_received_risk,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "displaylogo": False
            }
        )


    # =========================================================
    # NET INVENTORY EXPOSURE
    # =========================================================

    net_expired_quantity = max(
        expired_quantity - received_expired_quantity,
        0
    )

    net_expired_cost = max(
        expired_cost - received_expired_cost,
        0
    )

    net_at_risk_quantity = max(
        at_risk_quantity - received_at_risk_quantity,
        0
    )

    net_at_risk_cost = max(
        at_risk_cost - received_at_risk_cost,
        0
    )


    st.markdown(
        "<div style='height: 10px;'></div>",
        unsafe_allow_html=True
    )

    chart_title(
        "Net Inventory Exposure",
        "Inventory that remains exposed after excluding stock received already expired or at-risk"
    )

    net_col1, net_col2, net_col3, net_col4 = st.columns(4)


    with net_col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Net Expired Quantity
                </div>
                <div class="kpi-value">
                    {number(net_expired_quantity)}
                </div>
                <div class="kpi-sub">
                    Actual expired stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with net_col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Net Expired Cost
                </div>
                <div class="kpi-value">
                    ${number(net_expired_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of actual expired stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with net_col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Net At-Risk Quantity
                </div>
                <div class="kpi-value">
                    {number(net_at_risk_quantity)}
                </div>
                <div class="kpi-sub">
                    Remaining at-risk stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with net_col4:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Net At-Risk Cost
                </div>
                <div class="kpi-value">
                    ${number(net_at_risk_cost)}
                </div>
                <div class="kpi-sub">
                    Cost of remaining at-risk stock
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


elif page == "Employees":

    # =========================================================
    # EMPLOYEES PAGE
    # =========================================================

    st.title("Employees")

    st.markdown(
        "<div style='height: 6px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # PREPARE DATA
    # =========================================================

    employee_data = employees.copy()
    prescription_data = prescriptions.copy()
    store_data = stores.copy()


    employee_data["salary"] = pd.to_numeric(
        employee_data["salary"],
        errors="coerce"
    ).fillna(0)


    employee_data["is_pharmacist"] = pd.to_numeric(
        employee_data["is_pharmacist"],
        errors="coerce"
    ).fillna(0)


    prescription_data["prescription_id"] = (
        prescription_data["prescription_id"]
        .astype(str)
        .str.strip()
    )


    # =========================================================
    # GLOBAL FILTERS
    # =========================================================

    section_header(
        "Employee Filters"
    )


    filter_col1, filter_col2 = st.columns(2)


    # ---------------------------------------------------------
    # STORE FILTER
    # ---------------------------------------------------------

    with filter_col1:

        employee_store_options = [
            "All Stores"
        ] + sorted(
            store_data["store_name"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        selected_employee_store = st.selectbox(
            "Store",
            employee_store_options,
            key="employees_store_filter"
        )


    # ---------------------------------------------------------
    # POSITION FILTER
    # ---------------------------------------------------------

    with filter_col2:

        employee_position_options = [
            "All Positions"
        ] + sorted(
            employee_data["position"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        selected_employee_position = st.selectbox(
            "Position",
            employee_position_options,
            key="employees_position_filter"
        )


    # =========================================================
    # APPLY STORE FILTER
    # =========================================================

    if selected_employee_store != "All Stores":

        selected_store_ids = store_data.loc[
            store_data["store_name"]
            .astype(str)
            == selected_employee_store,
            "store_id"
        ].tolist()


        employee_data = employee_data[
            employee_data["store_id"]
            .isin(selected_store_ids)
        ]


        prescription_data = prescription_data[
            prescription_data["store_id"]
            .isin(selected_store_ids)
        ]


    # =========================================================
    # EMPLOYEE OVERVIEW
    # =========================================================

    section_header(
        "Employee Overview"
    )


    total_employees = (
        employee_data["employee_id"]
        .nunique()
    )


    total_salaries = (
        employee_data["salary"]
        .sum()
    )


    average_salary = (
        employee_data["salary"].mean()
        if total_employees > 0
        else 0
    )


    employee_col1, employee_col2, employee_col3 = st.columns(3)


    with employee_col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Employees
                </div>
                <div class="kpi-value">
                    {number(total_employees)}
                </div>
                <div class="kpi-sub">
                    Employees across selected stores
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with employee_col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Salaries
                </div>
                <div class="kpi-value">
                    ${number(total_salaries)}
                </div>
                <div class="kpi-sub">
                    Total employee salary cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with employee_col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Average Salary
                </div>
                <div class="kpi-value">
                    ${number(average_salary)}
                </div>
                <div class="kpi-sub">
                    Average salary per employee
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # EMPLOYEE DISTRIBUTION
    # =========================================================

    section_header(
        "Employee Distribution"
    )


    employee_metric = st.selectbox(
        "Select Employee Metric",
        [
            "Employees",
            "Total Salaries",
            "Average Salary"
        ],
        key="employee_distribution_metric"
    )


    # =========================================================
    # APPLY POSITION FILTER
    # ONLY FOR EMPLOYEE DISTRIBUTION
    # =========================================================

    employee_distribution_data = employee_data.copy()


    if selected_employee_position != "All Positions":

        employee_distribution_data = (
            employee_distribution_data[
                employee_distribution_data["position"]
                .astype(str)
                == selected_employee_position
            ]
        )


    # =========================================================
    # EMPLOYEES BY STORE
    # =========================================================

    store_employee = (
        employee_distribution_data
        .groupby(
            "store_id",
            as_index=False
        )
        .agg(
            Employees=(
                "employee_id",
                "nunique"
            ),
            Total_Salaries=(
                "salary",
                "sum"
            ),
            Average_Salary=(
                "salary",
                "mean"
            )
        )
    )


    store_employee = store_employee.merge(
        store_data[
            [
                "store_id",
                "store_name"
            ]
        ].drop_duplicates("store_id"),
        on="store_id",
        how="left"
    )


    store_employee["store_name"] = (
        store_employee["store_name"]
        .fillna(
            store_employee["store_id"]
            .astype(str)
        )
    )


    employee_metric_column = {
        "Employees": "Employees",
        "Total Salaries": "Total_Salaries",
        "Average Salary": "Average_Salary"
    }[
        employee_metric
    ]


    employee_metric_title = {
        "Employees": "Employees",
        "Total Salaries": "Total Salaries",
        "Average Salary": "Average Salary"
    }[
        employee_metric
    ]


    employee_store_chart = (
        store_employee
        .sort_values(
            employee_metric_column,
            ascending=False
        )
    )


    chart_title(
        "Employees by Store",
        "Employee distribution and salary structure across stores"
    )


    fig_store_employees = go.Figure()


    fig_store_employees.add_trace(
        go.Bar(
            x=employee_store_chart["store_name"],
            y=employee_store_chart[
                employee_metric_column
            ],
            text=employee_store_chart[
                employee_metric_column
            ].apply(
                lambda x: (
                    f"{x:,.0f}"
                    if employee_metric == "Employees"
                    else f"${x:,.2f}"
                )
            ),
            textposition="outside",
            hovertemplate=(
                "%{x}<br>"
                + employee_metric_title
                + ": "
                + (
                    "%{y:,.0f}"
                    if employee_metric == "Employees"
                    else "$%{y:,.2f}"
                )
                + "<extra></extra>"
            )
        )
    )


    fig_store_employees.update_layout(
        height=400,
        margin=dict(
            l=10,
            r=20,
            t=20,
            b=15
        ),
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#e2e8f0"
        ),
        xaxis=dict(
            showgrid=False,
            title="Store"
        ),
        yaxis=dict(
            gridcolor="#243044",
            title=employee_metric_title
        ),
        showlegend=False
    )


    st.plotly_chart(
        fig_store_employees,
        use_container_width=True,
        config={
            "displayModeBar": True,
            "displaylogo": False
        }
    )


    # =========================================================
    # EMPLOYEES BY POSITION
    # =========================================================

    position_employee = (
        employee_distribution_data
        .groupby(
            "position",
            as_index=False
        )
        .agg(
            Employees=(
                "employee_id",
                "nunique"
            ),
            Total_Salaries=(
                "salary",
                "sum"
            ),
            Average_Salary=(
                "salary",
                "mean"
            )
        )
    )


    position_employee["position"] = (
        position_employee["position"]
        .fillna("Unknown")
        .astype(str)
    )


    position_employee_chart = (
        position_employee
        .sort_values(
            employee_metric_column,
            ascending=False
        )
    )


    chart_title(
        "Employees by Position",
        "Employee distribution and salary structure by position"
    )


    fig_position_employees = go.Figure()


    fig_position_employees.add_trace(
        go.Bar(
            x=position_employee_chart["position"],
            y=position_employee_chart[
                employee_metric_column
            ],
            text=position_employee_chart[
                employee_metric_column
            ].apply(
                lambda x: (
                    f"{x:,.0f}"
                    if employee_metric == "Employees"
                    else f"${x:,.2f}"
                )
            ),
            textposition="outside",
            hovertemplate=(
                "%{x}<br>"
                + employee_metric_title
                + ": "
                + (
                    "%{y:,.0f}"
                    if employee_metric == "Employees"
                    else "$%{y:,.2f}"
                )
                + "<extra></extra>"
            )
        )
    )


    fig_position_employees.update_layout(
        height=400,
        margin=dict(
            l=10,
            r=20,
            t=20,
            b=15
        ),
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#e2e8f0"
        ),
        xaxis=dict(
            showgrid=False,
            title="Position"
        ),
        yaxis=dict(
            gridcolor="#243044",
            title=employee_metric_title
        ),
        showlegend=False
    )


    st.plotly_chart(
        fig_position_employees,
        use_container_width=True,
        config={
            "displayModeBar": True,
            "displaylogo": False
        }
    )


    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # PHARMACY WORKFORCE
    # =========================================================

    section_header(
        "Pharmacy Workforce"
    )


    total_pharmacists = (
        employee_data.loc[
            employee_data["is_pharmacist"] == 1,
            "employee_id"
        ]
        .nunique()
    )


    total_prescriptions = (
        prescription_data["prescription_id"]
        .nunique()
    )


    pharmacy_col1, pharmacy_col2 = st.columns(2)


    with pharmacy_col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Pharmacists
                </div>
                <div class="kpi-value">
                    {number(total_pharmacists)}
                </div>
                <div class="kpi-sub">
                    Pharmacists across selected stores
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with pharmacy_col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Prescriptions
                </div>
                <div class="kpi-value">
                    {number(total_prescriptions)}
                </div>
                <div class="kpi-sub">
                    Prescriptions across selected stores
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        "<div style='height: 24px;'></div>",
        unsafe_allow_html=True
    )


    # =========================================================
    # PHARMACY WORKFORCE METRIC
    # =========================================================

    pharmacy_metric = st.selectbox(
        "Select Pharmacy Metric",
        [
            "Pharmacists",
            "Prescriptions",
            "Prescriptions per Pharmacist"
        ],
        key="pharmacy_workforce_metric"
    )


    # =========================================================
    # PHARMACISTS BY STORE
    # =========================================================

    pharmacist_by_store = (
        employee_data.loc[
            employee_data["is_pharmacist"] == 1
        ]
        .groupby(
            "store_id",
            as_index=False
        )
        .agg(
            Pharmacists=(
                "employee_id",
                "nunique"
            )
        )
    )


    # =========================================================
    # PRESCRIPTIONS BY STORE
    # =========================================================

    prescriptions_by_store = (
        prescription_data
        .groupby(
            "store_id",
            as_index=False
        )
        .agg(
            Prescriptions=(
                "prescription_id",
                "nunique"
            )
        )
    )


    # =========================================================
    # COMBINE STORE METRICS
    # =========================================================

    pharmacy_store = (
        store_data[
            [
                "store_id",
                "store_name"
            ]
        ]
        .drop_duplicates("store_id")
        .merge(
            pharmacist_by_store,
            on="store_id",
            how="left"
        )
        .merge(
            prescriptions_by_store,
            on="store_id",
            how="left"
        )
    )


    pharmacy_store["Pharmacists"] = (
        pharmacy_store["Pharmacists"]
        .fillna(0)
    )


    pharmacy_store["Prescriptions"] = (
        pharmacy_store["Prescriptions"]
        .fillna(0)
    )


    pharmacy_store[
        "Prescriptions_per_Pharmacist"
    ] = (
        pharmacy_store["Prescriptions"]
        /
        pharmacy_store["Pharmacists"]
        .replace(0, pd.NA)
    ).fillna(0)


    pharmacy_metric_column = {
        "Pharmacists": "Pharmacists",
        "Prescriptions": "Prescriptions",
        "Prescriptions per Pharmacist":
            "Prescriptions_per_Pharmacist"
    }[
        pharmacy_metric
    ]


    pharmacy_chart = (
        pharmacy_store
        .sort_values(
            pharmacy_metric_column,
            ascending=False
        )
    )


    # =========================================================
    # PHARMACY WORKFORCE CHART
    # =========================================================

    chart_title(
        "Pharmacy Workforce & Prescription Activity",
        "Pharmacists and prescription workload across stores"
    )


    fig_pharmacy = go.Figure()


    fig_pharmacy.add_trace(
        go.Bar(
            x=pharmacy_chart["store_name"],
            y=pharmacy_chart[
                pharmacy_metric_column
            ],
            text=pharmacy_chart[
                pharmacy_metric_column
            ].apply(
                lambda x: (
                    f"{x:,.2f}"
                    if pharmacy_metric ==
                    "Prescriptions per Pharmacist"
                    else f"{x:,.0f}"
                )
            ),
            textposition="outside",
            hovertemplate=(
                "%{x}<br>"
                + pharmacy_metric
                + ": "
                + (
                    "%{y:,.2f}"
                    if pharmacy_metric ==
                    "Prescriptions per Pharmacist"
                    else "%{y:,.0f}"
                )
                + "<extra></extra>"
            )
        )
    )


    fig_pharmacy.update_layout(
        height=420,
        margin=dict(
            l=10,
            r=20,
            t=20,
            b=15
        ),
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#e2e8f0"
        ),
        xaxis=dict(
            showgrid=False,
            title="Store"
        ),
        yaxis=dict(
            gridcolor="#243044",
            title=pharmacy_metric
        ),
        showlegend=False
    )


    st.plotly_chart(
        fig_pharmacy,
        use_container_width=True,
        config={
            "displayModeBar": True,
            "displaylogo": False
        }
    )

elif page == "Suppliers":

    # =========================================================
    # SUPPLIERS PAGE
    # =========================================================

    supplier_data = suppliers.copy()
    product_data = products.copy()
    stock_data = stock.copy()
    store_data = stores.copy()

    # =========================================================
    # DATA PREPARATION
    # =========================================================

    stock_data["quantity"] = pd.to_numeric(
        stock_data["quantity"],
        errors="coerce"
    ).fillna(0)

    product_data["cost_price"] = pd.to_numeric(
        product_data["cost_price"],
        errors="coerce"
    ).fillna(0)

    stock_data["received_date"] = pd.to_datetime(
        stock_data["received_date"],
        errors="coerce"
    )

    stock_data["expiry_date"] = pd.to_datetime(
        stock_data["expiry_date"],
        errors="coerce"
    )

    # =========================================================
    # GENERAL FILTERS
    # =========================================================

    filter_col1, filter_col2 = st.columns(2)

    with filter_col1:

        selected_supplier_store = st.selectbox(
            "Store",
            ["All Stores"] + sorted(
                store_data["store_name"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            ),
            key="supplier_store_filter"
        )

    with filter_col2:

        selected_supplier = st.selectbox(
            "Supplier",
            ["All Suppliers"] + sorted(
                supplier_data["supplier_name"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            ),
            key="supplier_filter"
        )

    # =========================================================
    # STORE FILTER
    # =========================================================

    if selected_supplier_store != "All Stores":

        selected_store_ids = store_data.loc[
            store_data["store_name"].astype(str)
            == selected_supplier_store,
            "store_id"
        ].tolist()

        stock_data = stock_data[
            stock_data["store_id"].isin(selected_store_ids)
        ]

    # =========================================================
    # SUPPLIER → PRODUCT → STOCK
    # =========================================================

    supplier_stock = (
        stock_data
        .merge(
            product_data[
                [
                    "product_id",
                    "product_name",
                    "supplier_id",
                    "cost_price"
                ]
            ],
            on="product_id",
            how="left"
        )
        .merge(
            supplier_data[
                [
                    "supplier_id",
                    "supplier_name"
                ]
            ],
            on="supplier_id",
            how="left"
        )
    )

    # =========================================================
    # SUPPLIER FILTER
    # =========================================================

    if selected_supplier != "All Suppliers":

        supplier_stock = supplier_stock[
            supplier_stock["supplier_name"].astype(str)
            == selected_supplier
        ]

    # =========================================================
    # SUPPLY COST
    # =========================================================

    supplier_stock["supply_cost"] = (
        supplier_stock["quantity"]
        * supplier_stock["cost_price"]
    )

    # =========================================================
    # EXPIRY / RISK LOGIC
    # =========================================================

    supplier_stock["received_expired"] = (
        supplier_stock["expiry_date"].notna()
        & supplier_stock["received_date"].notna()
        & (
            supplier_stock["received_date"]
            > supplier_stock["expiry_date"]
        )
        & (supplier_stock["quantity"] > 0)
    )

    supplier_stock["received_at_risk"] = (
        supplier_stock["expiry_date"].notna()
        & supplier_stock["received_date"].notna()
        & (
            supplier_stock["received_date"]
            >= (
                supplier_stock["expiry_date"]
                - pd.DateOffset(months=3)
            )
        )
        & (
            supplier_stock["received_date"]
            <= supplier_stock["expiry_date"]
        )
        & (supplier_stock["quantity"] > 0)
    )

    # ---------------------------------------------------------
    # EXPIRED
    # ---------------------------------------------------------

    supplier_stock["received_expired_quantity"] = (
        supplier_stock["quantity"].where(
            supplier_stock["received_expired"],
            0
        )
    )

    supplier_stock["received_expired_cost"] = (
        supplier_stock["supply_cost"].where(
            supplier_stock["received_expired"],
            0
        )
    )

    # ---------------------------------------------------------
    # AT-RISK
    # ---------------------------------------------------------

    supplier_stock["received_at_risk_quantity"] = (
        supplier_stock["quantity"].where(
            supplier_stock["received_at_risk"],
            0
        )
    )

    supplier_stock["received_at_risk_cost"] = (
        supplier_stock["supply_cost"].where(
            supplier_stock["received_at_risk"],
            0
        )
    )

    # =========================================================
    # SUPPLIER OVERVIEW
    # =========================================================

    section_header("Supplier Overview")

    total_suppliers = supplier_stock["supplier_id"].nunique()

    total_supply_quantity = supplier_stock["quantity"].sum()

    total_supply_cost = supplier_stock["supply_cost"].sum()

    col1, col2, col3 = st.columns(3)

    # ---------------------------------------------------------
    # TOTAL SUPPLIERS
    # ---------------------------------------------------------

    with col1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Total Suppliers</div>
                <div class="kpi-value">
                    {number(total_suppliers)}
                </div>
                <div class="kpi-sub">
                    Suppliers in filtered data
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    # ---------------------------------------------------------
    # TOTAL SUPPLY QUANTITY
    # ---------------------------------------------------------

    with col2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Total Supply Quantity</div>
                <div class="kpi-value">
                    {number(total_supply_quantity)}
                </div>
                <div class="kpi-sub">
                    Units supplied
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    # ---------------------------------------------------------
    # TOTAL SUPPLY COST
    # ---------------------------------------------------------

    with col3:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">Total Supply Cost</div>
                <div class="kpi-value">
                    ${number(total_supply_cost)}
                </div>
                <div class="kpi-sub">
                    Total supply cost
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown(
        "<div style='height: 32px;'></div>",
        unsafe_allow_html=True
    )

    # =========================================================
    # SUPPLIER SUPPLY
    # =========================================================

    section_header("Supplier Supply")

    supplier_metric = st.selectbox(
        "Supply Metric",
        [
            "Supply Quantity",
            "Supply Cost",
            "Products Supplied"
        ],
        key="supplier_supply_metric"
    )

    supplier_summary = (
        supplier_stock
        .groupby(
            ["supplier_id", "supplier_name"],
            as_index=False
        )
        .agg(
            Supply_Quantity=("quantity", "sum"),
            Supply_Cost=("supply_cost", "sum"),
            Products_Supplied=("product_id", "nunique")
        )
    )

    # ---------------------------------------------------------
    # SELECTED METRIC
    # ---------------------------------------------------------

    if supplier_metric == "Supply Quantity":

        supplier_value = "Supply_Quantity"
        supplier_y_title = "Supply Quantity"
        supplier_text_template = "%{text:,.0f}"

    elif supplier_metric == "Supply Cost":

        supplier_value = "Supply_Cost"
        supplier_y_title = "Supply Cost"
        supplier_text_template = "$%{text:,.2f}"

    else:

        supplier_value = "Products_Supplied"
        supplier_y_title = "Products Supplied"
        supplier_text_template = "%{text:,.0f}"

    supplier_summary = supplier_summary.sort_values(
        supplier_value,
        ascending=False
    )

    chart_title(
        "Supply by Supplier",
        "Select a metric to compare suppliers"
    )

    fig_supplier = go.Figure()

    fig_supplier.add_trace(
        go.Bar(
            x=supplier_summary["supplier_name"],
            y=supplier_summary[supplier_value],
            text=supplier_summary[supplier_value],
            texttemplate=supplier_text_template,
            textposition="outside",
            cliponaxis=False,

            customdata=supplier_summary[
                [
                    "Supply_Quantity",
                    "Supply_Cost",
                    "Products_Supplied"
                ]
            ],

            hovertemplate=(
                "<b>%{x}</b><br>"
                "Supply Quantity: %{customdata[0]:,.0f}<br>"
                "Supply Cost: $%{customdata[1]:,.2f}<br>"
                "Products Supplied: %{customdata[2]:,.0f}"
                "<extra></extra>"
            )
        )
    )

    fig_supplier.update_layout(
        height=440,

        margin=dict(
            l=10,
            r=20,
            t=45,
            b=15
        ),

        paper_bgcolor="#111827",
        plot_bgcolor="#111827",

        font=dict(
            color="#e2e8f0"
        ),

        xaxis=dict(
            showgrid=False,
            title=None
        ),

        yaxis=dict(
            showgrid=True,
            gridcolor="#243044",
            title=supplier_y_title
        ),

        showlegend=False
    )

    st.plotly_chart(
        fig_supplier,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )

    st.markdown(
        "<div style='height: 38px;'></div>",
        unsafe_allow_html=True
    )

    # =========================================================
    # SUPPLIER EXPIRY & RISK
    # =========================================================

    section_header("Supplier Expiry & Risk")

    expiry_col, risk_col = st.columns(2)

    # =========================================================
    # RECEIVED EXPIRED
    # =========================================================

    with expiry_col:

        expired_metric = st.selectbox(
            "Received Expired Metric",
            [
                "Quantity",
                "Cost"
            ],
            key="supplier_expired_metric"
        )

        expired_summary = (
            supplier_stock
            .groupby(
                ["supplier_id", "supplier_name"],
                as_index=False
            )
            .agg(
                Quantity=(
                    "received_expired_quantity",
                    "sum"
                ),
                Cost=(
                    "received_expired_cost",
                    "sum"
                )
            )
        )

        if expired_metric == "Quantity":

            expired_value = "Quantity"
            expired_y_title = "Quantity"
            expired_text_template = "%{text:,.0f}"

        else:

            expired_value = "Cost"
            expired_y_title = "Cost"
            expired_text_template = "$%{text:,.2f}"

        expired_summary = expired_summary.sort_values(
            expired_value,
            ascending=False
        )

        chart_title(
            "Received Expired",
            "Stock received after its expiry date"
        )

        fig_expired = go.Figure()

        fig_expired.add_trace(
            go.Bar(
                x=expired_summary["supplier_name"],
                y=expired_summary[expired_value],

                text=expired_summary[expired_value],

                texttemplate=expired_text_template,

                textposition="outside",

                cliponaxis=False,

                customdata=expired_summary[
                    [
                        "Quantity",
                        "Cost"
                    ]
                ],

                hovertemplate=(
                    "<b>%{x}</b><br>"
                    "Quantity: %{customdata[0]:,.0f}<br>"
                    "Cost: $%{customdata[1]:,.2f}"
                    "<extra></extra>"
                )
            )
        )

        fig_expired.update_layout(
            height=410,

            margin=dict(
                l=10,
                r=15,
                t=45,
                b=15
            ),

            paper_bgcolor="#111827",
            plot_bgcolor="#111827",

            font=dict(
                color="#e2e8f0"
            ),

            xaxis=dict(
                showgrid=False,
                title=None
            ),

            yaxis=dict(
                showgrid=True,
                gridcolor="#243044",
                title=expired_y_title
            ),

            showlegend=False
        )

        st.plotly_chart(
            fig_expired,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )

    # =========================================================
    # RECEIVED AT-RISK
    # =========================================================

    with risk_col:

        risk_metric = st.selectbox(
            "Received At-Risk Metric",
            [
                "Quantity",
                "Cost"
            ],
            key="supplier_at_risk_metric"
        )

        risk_summary = (
            supplier_stock
            .groupby(
                ["supplier_id", "supplier_name"],
                as_index=False
            )
            .agg(
                Quantity=(
                    "received_at_risk_quantity",
                    "sum"
                ),
                Cost=(
                    "received_at_risk_cost",
                    "sum"
                )
            )
        )

        if risk_metric == "Quantity":

            risk_value = "Quantity"
            risk_y_title = "Quantity"
            risk_text_template = "%{text:,.0f}"

        else:

            risk_value = "Cost"
            risk_y_title = "Cost"
            risk_text_template = "$%{text:,.2f}"

        risk_summary = risk_summary.sort_values(
            risk_value,
            ascending=False
        )

        chart_title(
            "Received At-Risk",
            "Stock received within the final 3 months before expiry"
        )

        fig_risk = go.Figure()

        fig_risk.add_trace(
            go.Bar(
                x=risk_summary["supplier_name"],
                y=risk_summary[risk_value],

                text=risk_summary[risk_value],

                texttemplate=risk_text_template,

                textposition="outside",

                cliponaxis=False,

                customdata=risk_summary[
                    [
                        "Quantity",
                        "Cost"
                    ]
                ],

                hovertemplate=(
                    "<b>%{x}</b><br>"
                    "Quantity: %{customdata[0]:,.0f}<br>"
                    "Cost: $%{customdata[1]:,.2f}"
                    "<extra></extra>"
                )
            )
        )

        fig_risk.update_layout(
            height=410,

            margin=dict(
                l=10,
                r=15,
                t=45,
                b=15
            ),

            paper_bgcolor="#111827",
            plot_bgcolor="#111827",

            font=dict(
                color="#e2e8f0"
            ),

            xaxis=dict(
                showgrid=False,
                title=None
            ),

            yaxis=dict(
                showgrid=True,
                gridcolor="#243044",
                title=risk_y_title
            ),

            showlegend=False
        )

        st.plotly_chart(
            fig_risk,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )

    st.markdown(
        "<div style='height: 28px;'></div>",
        unsafe_allow_html=True
    )

In [ ]:
import os

os.makedirs("dashboard_data", exist_ok=True)

dataframes = {
    "categories": categories,
    "customers": customers,
    "employees": employees,
    "prescriptions": prescriptions,
    "products": products,
    "stock": stock,
    "suppliers": suppliers,
    "stores": stores,
    "sales": sales,
}

for name, df in dataframes.items():
    df.to_parquet(f"dashboard_data/{name}.parquet", index=False)

print("Dashboard data exported successfully.")